## Problem Formulation

**Binary classification of GPS-observation / candidate-road-segment pairs.**

For each GPS observation we generate K candidate road segments. The classifier learns to assign a high positive-class probability to the correct road and a low probability to incorrect roads.

**Decision at inference time:**  
For each GPS observation, the candidate with the **highest positive-class probability** is selected as the Top-1 predicted road segment, regardless of whether that probability exceeds the 0.5 threshold.

The 0.5 threshold is only used for binary candidate-level metrics (Accuracy, Precision, Recall, Macro F1).

In [1]:
# Cell 3 – Library imports
import sys, json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, log_loss,
    classification_report
)
from sklearn.model_selection import GroupKFold
from sklearn.inspection import permutation_importance

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
print('Libraries imported successfully.')

Libraries imported successfully.


In [2]:
# Cell 4 – Project paths
import sys
from pathlib import Path

current_dir = Path.cwd()
if (current_dir / 'data' / 'processed').exists():
    PROJECT_ROOT = current_dir
elif (current_dir / 'Random Forest' / 'data' / 'processed').exists():
    PROJECT_ROOT = current_dir / 'Random Forest'
elif (current_dir.parent / 'data' / 'processed').exists():
    PROJECT_ROOT = current_dir.parent
elif (current_dir.parent / 'Random Forest' / 'data' / 'processed').exists():
    PROJECT_ROOT = current_dir.parent / 'Random Forest'
else:
    found = list(current_dir.glob('**/train_candidates.parquet'))
    PROJECT_ROOT = found[0].parent.parent.parent if found else current_dir

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if (PROJECT_ROOT / 'shared').exists() and str(PROJECT_ROOT / 'shared') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'shared'))

# Data paths
PROC_DIR   = PROJECT_ROOT / 'data' / 'processed'
TRAIN_PATH = PROC_DIR / 'train_candidates.parquet'
TEST_PATH  = PROC_DIR / 'student_test_candidates.parquet'
FOLD_PATH  = PROC_DIR / 'fold_assignments.csv'

# Model paths
RF_DIR        = PROJECT_ROOT / 'models' / 'random_forest'
if not RF_DIR.exists() and (PROJECT_ROOT / 'random_forest').exists():
    RF_DIR = PROJECT_ROOT / 'random_forest'

ARTIFACTS_DIR = RF_DIR / 'artifacts'
RESULTS_DIR   = RF_DIR / 'results'
PLOTS_DIR     = RF_DIR / 'plots'

for d in [ARTIFACTS_DIR, RESULTS_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Project paths:')
for name, p in [('Train features', TRAIN_PATH), ('Test features', TEST_PATH),
                ('Fold assignments', FOLD_PATH), ('Artifacts', ARTIFACTS_DIR),
                ('Results', RESULTS_DIR), ('Plots', PLOTS_DIR)]:
    exists = '✓ exists' if p.exists() else '✗ NOT FOUND'
    print(f'  {name:20s}: {p}  [{exists}]')


Project paths:
  Train features      : /Users/sandeeph/Desktop/ML_Lab/Random Forest/data/processed/train_candidates.parquet  [✓ exists]
  Test features       : /Users/sandeeph/Desktop/ML_Lab/Random Forest/data/processed/student_test_candidates.parquet  [✓ exists]
  Fold assignments    : /Users/sandeeph/Desktop/ML_Lab/Random Forest/data/processed/fold_assignments.csv  [✓ exists]
  Artifacts           : /Users/sandeeph/Desktop/ML_Lab/Random Forest/models/random_forest/artifacts  [✓ exists]
  Results             : /Users/sandeeph/Desktop/ML_Lab/Random Forest/models/random_forest/results  [✓ exists]
  Plots               : /Users/sandeeph/Desktop/ML_Lab/Random Forest/models/random_forest/plots  [✓ exists]


In [3]:
# Cell 5 – Reproducibility settings
RANDOM_STATE = 42
N_FOLDS      = 5
THRESHOLD    = 0.5

np.random.seed(RANDOM_STATE)

import sklearn
print(f'Python version:         {sys.version.split()[0]}')
print(f'NumPy version:          {np.__version__}')
print(f'Pandas version:         {pd.__version__}')
print(f'Scikit-learn version:   {sklearn.__version__}')
print(f'Random state:           {RANDOM_STATE}')
print(f'CV folds:               {N_FOLDS}')
print(f'Decision threshold:     {THRESHOLD}')

Python version:         3.14.6
NumPy version:          2.4.6
Pandas version:         3.0.3
Scikit-learn version:   1.9.0
Random state:           42
CV folds:               5
Decision threshold:     0.5


## Common Input-Data Contract

The common preprocessing notebook (Notebook 00) must produce a candidate-level dataset with the following columns:

| Column | Type | Description |
|---|---|---|
| `trip_id` | str | Trip identifier (metadata only) |
| `observation_id` | str | GPS observation identifier (metadata only) |
| `candidate_arc_id` | int | Candidate road segment ID (metadata only) |
| `reference_arc_id` | int | Reference road segment from alignment (metadata only) |
| `label` | int | 1 = reference candidate, 0 = incorrect candidate |
| `perpendicular_distance` | float | GPS-to-road distance (m) |
| `projection_fraction` | float | Position along arc [0,1] |
| ... (15 features total) | | See Section C |

> **Never used as features:** `trip_id`, `observation_id`, `candidate_arc_id`, `reference_arc_id`, `label`

In [4]:
# Cell 7 – Load training candidate dataset
FEATURE_COLS = [
    'perpendicular_distance', 'projection_fraction', 'distance_to_start',
    'distance_to_end', 'segment_length', 'road_bearing', 'gps_bearing',
    'heading_difference', 'gps_step_distance', 'time_difference',
    'estimated_speed', 'candidate_rank', 'candidate_count',
    'gps_gap_flag', 'heading_valid_flag',
]
METADATA_COLS = ['trip_id', 'observation_id', 'candidate_arc_id',
                 'reference_arc_id', 'label']

if TRAIN_PATH.exists():
    train_df = pd.read_parquet(TRAIN_PATH)
    print(f'Training data loaded: {train_df.shape}')
else:
    print('WARNING: train_candidates.parquet not found. Generating synthetic demo data.')
    np.random.seed(42)
    rows = []
    for t in range(80):
        tid = f'T{t+1:02d}'
        for o in range(25):
            oid = f'{tid}_OBS_{o:03d}'
            ref = int(np.random.randint(0, 150))
            pos_rank = int(np.random.randint(1, 9))
            for rank in range(1, 9):
                rows.append({
                    'trip_id': tid, 'observation_id': oid,
                    'candidate_arc_id': (ref + rank - 1) % 150,
                    'reference_arc_id': ref, 'label': int(rank == pos_rank),
                    'perpendicular_distance': float(np.random.exponential(5)*rank),
                    'projection_fraction': float(np.random.uniform(0,1)),
                    'distance_to_start': float(np.random.uniform(0,100)),
                    'distance_to_end': float(np.random.uniform(0,100)),
                    'segment_length': float(np.random.uniform(20,200)),
                    'road_bearing': float(np.random.uniform(0,360)),
                    'gps_bearing': float(np.random.uniform(0,360)) if o>0 else np.nan,
                    'heading_difference': float(np.random.uniform(0,180)) if o>0 else np.nan,
                    'gps_step_distance': float(np.random.uniform(0,30)) if o>0 else np.nan,
                    'time_difference': float(np.random.uniform(1,20)) if o>0 else np.nan,
                    'estimated_speed': float(np.random.uniform(0,15)) if o>0 else np.nan,
                    'candidate_rank': rank, 'candidate_count': 8,
                    'gps_gap_flag': int(np.random.rand() < 0.05),
                    'heading_valid_flag': int(o>0 and np.random.rand()<0.9),
                    'gps_fallback_flag': 0,
                })
    train_df = pd.DataFrame(rows)
    print(f'Synthetic training data: {train_df.shape}')

Training data loaded: (1127273, 21)


In [5]:
# Cell 8 – Load student-testing dataset
if TEST_PATH.exists():
    test_df = pd.read_parquet(TEST_PATH)
    print(f'Student-test data loaded: {test_df.shape}')
else:
    print('WARNING: student_test_candidates.parquet not found. Using last 10 training trips as proxy.')
    last_trips = sorted(train_df['trip_id'].unique())[-10:]
    test_df = train_df[train_df['trip_id'].isin(last_trips)].copy()
    print(f'Proxy student-test data: {test_df.shape}')

print(f'\nStudent-test trips: {sorted(test_df["trip_id"].unique())}')

Student-test data loaded: (122990, 21)

Student-test trips: ['00000000', '00000007', '00000009', '00000040', '00000052', '00000056', '00000065', '00000076', '00000077', '00000091']


In [6]:
# Cell 9 – Dataset overview
print('=== TRAINING DATASET ===')
print(f'Shape: {train_df.shape}')
print(f'Trips: {train_df["trip_id"].nunique()}')
print(f'\nColumn dtypes:')
print(train_df.dtypes)
print('\nFirst 5 rows:')
display(train_df.head())

=== TRAINING DATASET ===
Shape: (1127273, 21)
Trips: 80

Column dtypes:
trip_id                       str
observation_id                str
candidate_arc_id            int64
reference_arc_id            int64
label                       int64
perpendicular_distance    float32
projection_fraction       float32
distance_to_start         float32
distance_to_end           float32
segment_length            float32
road_bearing              float32
gps_bearing               float32
heading_difference        float32
gps_step_distance         float32
time_difference           float32
estimated_speed           float32
candidate_rank              int32
candidate_count             int32
gps_gap_flag                int32
heading_valid_flag          int32
gps_fallback_flag           int32
dtype: object

First 5 rows:


,trip_id,observation_id,candidate_arc_id,reference_arc_id,label,perpendicular_distance,projection_fraction,distance_to_start,distance_to_end,segment_length,...,gps_bearing,heading_difference,gps_step_distance,time_difference,estimated_speed,candidate_rank,candidate_count,gps_gap_flag,heading_valid_flag,gps_fallback_flag
0,00000001,00000001_OBS_0000,64973,64973,1,0.722768,0.109775,32.198444,261.048615,293.237946,...,NaN,NaN,NaN,NaN,NaN,1,3,0,0,0
1,00000001,00000001_OBS_0000,117639,64973,0,15.132321,0.512692,523.389282,497.497803,1020.438110,...,NaN,NaN,NaN,NaN,NaN,2,3,0,0,0
2,00000001,00000001_OBS_0000,2641,64973,0,32.198444,1.000000,457.242645,32.198444,425.050293,...,NaN,NaN,NaN,NaN,NaN,3,3,0,0,0
3,00000001,00000001_OBS_0001,64973,64973,1,0.277461,0.258531,75.811623,217.427002,293.237946,...,8.933595,1.313567,43.632248,1.0,43.632248,1,2,0,1,0
4,00000001,00000001_OBS_0001,117639,64973,0,16.079744,0.469944,479.818054,541.128540,1020.438110,...,8.933595,178.755798,43.632248,1.0,43.632248,2,2,0,1,0


In [7]:
# Cell 10 – Column verification
expected_cols = set(METADATA_COLS + FEATURE_COLS)
present_cols  = set(train_df.columns)
missing_in_train = expected_cols - present_cols
extra_in_train   = present_cols - expected_cols - {'gps_fallback_flag'}

print('Column verification:')
print(f'  Expected columns:  {len(expected_cols)}')
print(f'  Present columns:   {len(present_cols)}')
print(f'  Missing:           {missing_in_train if missing_in_train else "None"}')
print(f'  Extra (OK):        {extra_in_train}')

# Duplicate candidate check
n_dup = train_df.duplicated(subset=['observation_id', 'candidate_arc_id']).sum()
print(f'\nDuplicate obs+arc rows: {n_dup}')
assert n_dup == 0 or print('WARNING: Duplicates found!')

# Check no training trip in test set
train_trips = set(train_df['trip_id'].unique())
test_trips  = set(test_df['trip_id'].unique())
overlap = train_trips & test_trips
print(f'\nTrain/test trip overlap: {overlap if overlap else "None ✓"}')

Column verification:
  Expected columns:  20
  Present columns:   21
  Missing:           None
  Extra (OK):        set()

Duplicate obs+arc rows: 0

Train/test trip overlap: None ✓


In [8]:
# Cell 11 – Missing and infinite value inspection
print('=== MISSING VALUE REPORT ===')
mv_rows = []
for col in FEATURE_COLS:
    if col not in train_df.columns:
        continue
    n_miss = train_df[col].isna().sum()
    n_inf  = np.isinf(train_df[col].replace(np.nan, 0)).sum()
    mv_rows.append({'feature': col, 'n_missing': n_miss,
                    'pct_missing': round(100*n_miss/len(train_df),2), 'n_inf': n_inf})

mv_df = pd.DataFrame(mv_rows)
print(mv_df[mv_df['n_missing'] > 0].to_string(index=False) if (mv_df['n_missing']>0).any()
      else 'No missing values in numeric features (or only expected NaN in heading/speed for first obs)')
print('\nNote: NaN in gps_bearing, heading_difference, gps_step_distance, time_difference,')
print('      estimated_speed for first observation of each trip is EXPECTED and will be imputed.')

=== MISSING VALUE REPORT ===
               feature  n_missing  pct_missing  n_inf
perpendicular_distance       1645         0.15      0
   projection_fraction       1645         0.15      0
     distance_to_start       1645         0.15      0
       distance_to_end       1645         0.15      0
        segment_length       1645         0.15      0
          road_bearing       1645         0.15      0
           gps_bearing      25380         2.25      0
    heading_difference      25380         2.25      0
     gps_step_distance       2234         0.20      0
       time_difference        590         0.05      0
       estimated_speed       2234         0.20      0

Note: NaN in gps_bearing, heading_difference, gps_step_distance, time_difference,
      estimated_speed for first observation of each trip is EXPECTED and will be imputed.


In [9]:
# Cell 12 – Trip coverage verification
all_trips_in_train = sorted(train_df['trip_id'].unique())
all_trips_in_test  = sorted(test_df['trip_id'].unique())
print(f'Training trips ({len(all_trips_in_train)}): {all_trips_in_train[:5]} ... {all_trips_in_train[-3:]}')
print(f'Student-test trips ({len(all_trips_in_test)}): {all_trips_in_test}')

overlap = set(all_trips_in_train) & set(all_trips_in_test)
if overlap:
    print(f'\nWARNING: Overlap found: {overlap}')
else:
    print('\n✓ No trip overlap between training and student-testing partitions.')

Training trips (80): ['00000001', '00000002', '00000003', '00000004', '00000005'] ... ['00000097', '00000098', '00000099']
Student-test trips (10): ['00000000', '00000007', '00000009', '00000040', '00000052', '00000056', '00000065', '00000076', '00000077', '00000091']

✓ No trip overlap between training and student-testing partitions.


In [10]:
# Cell 13 – Per-trip observation and candidate counts
trip_stats = (train_df.groupby('trip_id')
              .agg(n_candidates=('candidate_arc_id','count'),
                   n_observations=('observation_id','nunique'))
              .reset_index())
trip_stats['avg_cands_per_obs'] = (trip_stats['n_candidates'] /
                                   trip_stats['n_observations']).round(2)

print(f'Per-trip statistics:')
print(trip_stats.describe().round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(trip_stats['n_observations'], bins=20, color='steelblue', edgecolor='white')
axes[0].set_xlabel('GPS Observations per Trip')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Observations per Trip')

axes[1].hist(trip_stats['avg_cands_per_obs'], bins=20, color='salmon', edgecolor='white')
axes[1].set_xlabel('Avg Candidates per Observation')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of Avg Candidates per Observation')

plt.tight_layout()
fig.savefig(PLOTS_DIR / 'trip_obs_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'\nTotal observations: {train_df["observation_id"].nunique():,}')
print(f'Total candidate rows: {len(train_df):,}')

Per-trip statistics:
       n_candidates  n_observations  avg_cands_per_obs
count         80.00           80.00              80.00
mean       14090.91         2605.21               6.67
std        11962.79         3312.35               1.98
min         1645.00          361.00               1.00
25%         6345.50          945.75               5.83
50%        11199.00         1529.00               6.95
75%        16551.00         2792.75               8.24
max        61221.00        19313.00               9.68

Total observations: 208,417
Total candidate rows: 1,127,273


In [11]:
# Cell 14 – Binary label verification
n_total = len(train_df)
n_pos   = (train_df['label'] == 1).sum()
n_neg   = (train_df['label'] == 0).sum()
print(f'Total candidate rows:    {n_total:,}')
print(f'Positive (label=1):      {n_pos:,}  ({100*n_pos/n_total:.2f}%)')
print(f'Negative (label=0):      {n_neg:,}  ({100*n_neg/n_total:.2f}%)')
print(f'Imbalance ratio (neg/pos): {n_neg/n_pos:.1f}:1')

# Observations with no positive candidate
obs_max_label = train_df.groupby('observation_id')['label'].max()
no_pos = (obs_max_label == 0).sum()
print(f'\nObservations with NO positive candidate: {no_pos} ({100*no_pos/len(obs_max_label):.2f}%)')
if no_pos > 0:
    print('WARNING: These will count as candidate-generation misses.')

# More than one positive per obs (should be rare / 0)
multi_pos = (train_df.groupby('observation_id')['label'].sum() > 1).sum()
print(f'Observations with >1 positive candidate: {multi_pos}')

Total candidate rows:    1,127,273
Positive (label=1):      181,770  (16.12%)
Negative (label=0):      945,503  (83.88%)
Imbalance ratio (neg/pos): 5.2:1

Observations with NO positive candidate: 26647 (12.79%)
Observations with >1 positive candidate: 0


## Dataset Verification – Outcome

The loaded dataset has been verified for:
- Column presence (15 features + metadata columns)
- Duplicate candidate rows (none expected)
- Training / student-testing partition overlap (none permitted)
- Binary label validity
- Candidate-generation miss rate (observations with no positive label)

Any data-quality issues should be resolved in Notebook 00 before proceeding.

In [12]:
# Cell 16 – Define the 15 agreed features
FEATURE_COLS = [
    'perpendicular_distance',   # 1  GPS-to-road perpendicular distance
    'projection_fraction',      # 2  relative position along segment [0,1]
    'distance_to_start',        # 3  distance to arc start node
    'distance_to_end',          # 4  distance to arc end node
    'segment_length',           # 5  physical arc length
    'road_bearing',             # 6  arc direction (degrees)
    'gps_bearing',              # 7  estimated vehicle heading
    'heading_difference',       # 8  abs angle GPS↔arc (0–180°)
    'gps_step_distance',        # 9  movement since previous observation
    'time_difference',          # 10 elapsed time from previous observation
    'estimated_speed',          # 11 estimated vehicle speed (m/s)
    'candidate_rank',           # 12 rank by perpendicular distance (1=closest)
    'candidate_count',          # 13 total candidates for this GPS observation
    'gps_gap_flag',             # 14 large GPS sampling gap (0/1)
    'heading_valid_flag',       # 15 reliable GPS heading available (0/1)
]
print(f'{len(FEATURE_COLS)} features defined.')
for i, f in enumerate(FEATURE_COLS, 1):
    print(f'  {i:2d}. {f}')

15 features defined.
   1. perpendicular_distance
   2. projection_fraction
   3. distance_to_start
   4. distance_to_end
   5. segment_length
   6. road_bearing
   7. gps_bearing
   8. heading_difference
   9. gps_step_distance
  10. time_difference
  11. estimated_speed
  12. candidate_rank
  13. candidate_count
  14. gps_gap_flag
  15. heading_valid_flag


In [13]:
# Cell 17 – Metadata columns (never enter the model)
METADATA_COLS = ['trip_id', 'observation_id', 'candidate_arc_id',
                 'reference_arc_id', 'label']
TARGET_COL = 'label'
print('Metadata columns (excluded from feature matrix):')
for c in METADATA_COLS:
    print(f'  {c}')

Metadata columns (excluded from feature matrix):
  trip_id
  observation_id
  candidate_arc_id
  reference_arc_id
  label


In [14]:
# Cell 18 – Create feature matrices
# Only keep features that are actually in the DataFrame
present_feats = [f for f in FEATURE_COLS if f in train_df.columns]
missing_feats = [f for f in FEATURE_COLS if f not in train_df.columns]
if missing_feats:
    print(f'WARNING: Missing features (will be filled with NaN): {missing_feats}')
    for f in missing_feats:
        train_df[f] = np.nan
        test_df[f]  = np.nan

X_train = train_df[FEATURE_COLS].to_numpy(dtype=np.float64)
y_train = train_df[TARGET_COL].to_numpy()
X_test  = test_df[FEATURE_COLS].to_numpy(dtype=np.float64)
y_test  = test_df[TARGET_COL].to_numpy()

meta_train = train_df[METADATA_COLS].copy()
meta_test  = test_df[METADATA_COLS].copy()

print(f'X_train shape: {X_train.shape}')
print(f'y_train shape: {y_train.shape}  positives: {y_train.sum():,}')
print(f'X_test  shape: {X_test.shape}')
print(f'y_test  shape: {y_test.shape}  positives: {y_test.sum():,}')

X_train shape: (1127273, 15)
y_train shape: (1127273,)  positives: 181,770
X_test  shape: (122990, 15)
y_test  shape: (122990,)  positives: 19,107


In [15]:
# Cell 19 – Feature distributions and correlation heatmap
fig, axes = plt.subplots(3, 5, figsize=(18, 10))
for ax, feat in zip(axes.flat, FEATURE_COLS):
    if feat in train_df.columns:
        vals = train_df[feat].dropna()
        ax.hist(vals, bins=40, color='steelblue', edgecolor='none', alpha=0.8)
    ax.set_title(feat, fontsize=8)
    ax.tick_params(labelsize=7)
plt.suptitle('Feature Distributions (Training Data)', fontsize=12, y=1.01)
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'feature_distributions.png', dpi=120, bbox_inches='tight')
plt.show()

# Correlation heatmap
feat_df = train_df[FEATURE_COLS].copy()
corr = feat_df.corr(numeric_only=True)
fig2, ax2 = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            annot=True, fmt='.2f', annot_kws={'size':7},
            linewidths=0.5, ax=ax2)
ax2.set_title('Feature Correlation Heatmap (Training Data)')
plt.tight_layout()
fig2.savefig(PLOTS_DIR / 'feature_correlation.png', dpi=120, bbox_inches='tight')
plt.show()

In [16]:
# Cell 20 – Class distribution plot
n_pos = int((y_train == 1).sum())
n_neg = int((y_train == 0).sum())
ratio = n_neg / n_pos if n_pos > 0 else float('inf')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Overall
bars = axes[0].bar(['Incorrect (0)', 'Correct (1)'], [n_neg, n_pos],
                   color=['#E06C75', '#98C379'], edgecolor='white')
for bar, v in zip(bars, [n_neg, n_pos]):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+50,
                 f'{v:,}', ha='center', fontsize=10)
axes[0].set_title(f'Class Distribution (imbalance ratio {ratio:.1f}:1)')
axes[0].set_ylabel('Count')

# Per-trip positive rate
trip_pos_rate = train_df.groupby('trip_id')['label'].mean()
axes[1].hist(trip_pos_rate, bins=20, color='steelblue', edgecolor='white')
axes[1].axvline(trip_pos_rate.mean(), color='crimson', linestyle='--',
                label=f'Mean = {trip_pos_rate.mean():.3f}')
axes[1].set_xlabel('Positive Rate per Trip')
axes[1].set_ylabel('Number of Trips')
axes[1].set_title('Per-Trip Class Imbalance Distribution')
axes[1].legend()

plt.tight_layout()
fig.savefig(PLOTS_DIR / 'class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Overall imbalance ratio: {ratio:.1f} negatives per positive')
print(f'  → Using class_weight="balanced_subsample" to address per-bag imbalance')

Overall imbalance ratio: 5.2 negatives per positive
  → Using class_weight="balanced_subsample" to address per-bag imbalance


In [17]:
# Cell 21 – Candidate Miss Rate
obs_groups = train_df.groupby('observation_id')
n_obs   = len(obs_groups)
n_miss  = sum(1 for _, g in obs_groups if (g['label']==1).sum() == 0)
miss_rate = n_miss / n_obs if n_obs > 0 else 0.0

print('=== CANDIDATE MISS RATE ===')
print(f'Total GPS observations:          {n_obs:,}')
print(f'Observations with correct road:  {n_obs-n_miss:,}')
print(f'Observations missing ref. road:  {n_miss:,}')
print(f'Candidate Miss Rate:             {miss_rate:.4f} ({100*miss_rate:.2f}%)')
print()
if miss_rate > 0.05:
    print('WARNING: Miss rate > 5% — consider increasing search radius or K.')
else:
    print('✓ Miss rate is acceptable.')

=== CANDIDATE MISS RATE ===
Total GPS observations:          208,417
Observations with correct road:  181,770
Observations missing ref. road:  26,647
Candidate Miss Rate:             0.1279 (12.79%)



## Feature Selection Summary

**Selected features:** 15 (see Cell 16)  
**Excluded metadata:** `trip_id`, `observation_id`, `candidate_arc_id`, `reference_arc_id`, `label`

**Why no StandardScaler?**  
Random Forest is a tree-based method. It makes binary split decisions at each node and is therefore **invariant to monotonic feature scaling**. Applying StandardScaler or MinMaxScaler would have zero effect on the trees' splitting decisions or the final predictions. We use `SimpleImputer(strategy='median')` only to handle the expected NaN values in kinematic features (first GPS observation of each trip has no previous point for heading/speed calculation).

## Random Forest Algorithm

**Decision Trees with Bootstrap Aggregation (Bagging):**

1. For each of the 400 trees, draw a bootstrap sample (80% of training rows with replacement)
2. At each node, consider only √(n_features) randomly chosen features
3. Find the split that maximises Gini impurity reduction
4. Grow tree to max_depth=24 (or until stopping criteria)
5. Final prediction = average of all 400 trees' `predict_proba` outputs

**Why RF for candidate matching?**
- Naturally handles non-linear relationships (e.g. perpendicular distance thresholds differ by road type)
- Robust to irrelevant features and correlated inputs
- Provides calibrated-enough probabilities for HMM emission scores
- No feature scaling required
- `balanced_subsample` corrects class imbalance within each tree's bootstrap sample

In [18]:
# Cell 24 – Initial Random Forest configuration (Phase 6.1)
RF_INITIAL_PARAMS = dict(
    n_estimators      = 400,
    criterion         = 'gini',
    max_depth         = 24,
    min_samples_split = 5,
    min_samples_leaf  = 2,
    max_features      = 'sqrt',
    bootstrap         = True,
    max_samples       = 0.8,
    class_weight      = 'balanced_subsample',
    n_jobs            = -1,
    random_state      = RANDOM_STATE,
)

print('Initial Random Forest Configuration:')
for k, v in RF_INITIAL_PARAMS.items():
    print(f'  {k:25s}: {v}')
print()
print('NOTE: This is the initial experiment configuration.')
print('      Bayesian optimisation (Phase 11) is conditional on model selection.')

Initial Random Forest Configuration:
  n_estimators             : 400
  criterion                : gini
  max_depth                : 24
  min_samples_split        : 5
  min_samples_leaf         : 2
  max_features             : sqrt
  bootstrap                : True
  max_samples              : 0.8
  class_weight             : balanced_subsample
  n_jobs                   : -1
  random_state             : 42

NOTE: This is the initial experiment configuration.
      Bayesian optimisation (Phase 11) is conditional on model selection.


In [19]:
# Cell 25 – Build sklearn Pipeline (imputer + RF, NO scaler)
def _to_float32(X):
    return X.astype(np.float32)

def build_rf_pipeline(params=None):
    if params is None:
        params = RF_INITIAL_PARAMS
    
    return Pipeline([
        ('imputer',    SimpleImputer(strategy='median', add_indicator=True)),
        ('to_float32', FunctionTransformer(_to_float32, validate=False)),
        ('classifier', RandomForestClassifier(**params)),
    ])

pipeline = build_rf_pipeline()
print('Pipeline structure:')
for name, step in pipeline.steps:
    print(f'  {name}: {type(step).__name__}')
print()
print('Note: add_indicator=True preserves missingness information as binary columns.')
print('Note: NO StandardScaler — RF is scale-invariant by design.')


Pipeline structure:
  imputer: SimpleImputer
  to_float32: FunctionTransformer
  classifier: RandomForestClassifier

Note: add_indicator=True preserves missingness information as binary columns.
Note: NO StandardScaler — RF is scale-invariant by design.


In [20]:
# Cell 26 – Inspect pipeline parameters
print('Complete pipeline parameters:')
params = pipeline.get_params()
for k, v in params.items():
    if '__' not in k or k.startswith('classifier__'):
        print(f'  {k}: {v}')

Complete pipeline parameters:
  memory: None
  steps: [('imputer', SimpleImputer(add_indicator=True, strategy='median')), ('to_float32', FunctionTransformer(func=<function _to_float32 at 0x7098b8e8d0>)), ('classifier', RandomForestClassifier(class_weight='balanced_subsample', max_depth=24,
                       max_samples=0.8, min_samples_leaf=2, min_samples_split=5,
                       n_estimators=400, n_jobs=-1, random_state=42))]
  transform_input: None
  verbose: False
  imputer: SimpleImputer(add_indicator=True, strategy='median')
  to_float32: FunctionTransformer(func=<function _to_float32 at 0x7098b8e8d0>)
  classifier: RandomForestClassifier(class_weight='balanced_subsample', max_depth=24,
                       max_samples=0.8, min_samples_leaf=2, min_samples_split=5,
                       n_estimators=400, n_jobs=-1, random_state=42)
  classifier__bootstrap: True
  classifier__ccp_alpha: 0.0
  classifier__class_weight: balanced_subsample
  classifier__criterion: gini
 

In [21]:
# Cell 27 – Pipeline smoke test
print('Running pipeline smoke test on 10% of training data...')
smoke_n = max(500, len(train_df) // 10)
X_smoke = X_train[:smoke_n]
y_smoke = y_train[:smoke_n]

smoke_pipeline = build_rf_pipeline()
smoke_pipeline.fit(X_smoke, y_smoke)
proba_smoke = smoke_pipeline.predict_proba(X_smoke)

rf_smoke = smoke_pipeline.named_steps['classifier']
print(f'  Fitted classes:         {list(rf_smoke.classes_)}')
print(f'  predict_proba shape:    {proba_smoke.shape}  (expected: (n, 2))')
print(f'  Positive class index:   {list(rf_smoke.classes_).index(1)}')
print(f'  Prob range (pos class): [{proba_smoke[:,1].min():.4f}, {proba_smoke[:,1].max():.4f}]')
assert proba_smoke.shape[1] == 2, 'Expected 2-class output'
assert list(rf_smoke.classes_) == [0, 1] or set(rf_smoke.classes_) == {0, 1}
print()
print('✓ Smoke test PASSED. Pipeline is functional.')
del smoke_pipeline

Running pipeline smoke test on 10% of training data...
  Fitted classes:         [np.int64(0), np.int64(1)]
  predict_proba shape:    (112727, 2)  (expected: (n, 2))
  Positive class index:   1
  Prob range (pos class): [0.0000, 0.9999]

✓ Smoke test PASSED. Pipeline is functional.


## GroupKFold – Why Trip-Level Splits?

GPS observations from the same trip share spatial context, road network, and temporal sequence. Randomly splitting candidate rows would allow:
- Features from Trip X to appear in both training and validation
- The model to effectively "memorise" trip-specific patterns
- Inflated validation scores that don't generalise

**Solution:** Use `GroupKFold` with `trip_id` as the grouping variable. This ensures all candidates from a trip are either entirely in training or entirely in validation for each fold.

In [23]:
# Cell 29 – Load or generate 5-fold GroupKFold splits
if FOLD_PATH.exists():
    fold_df = pd.read_csv(FOLD_PATH, dtype={'trip_id': str, 'fold_id': int})
    sample_train_trip = str(train_df['trip_id'].iloc[0])
    if sample_train_trip.isdigit() and len(sample_train_trip) == 8:
        fold_df['trip_id'] = fold_df['trip_id'].apply(lambda x: f'{int(x):08d}')
    else:
        fold_df['trip_id'] = fold_df['trip_id'].astype(str)
    train_df['trip_id'] = train_df['trip_id'].astype(str)
    print(f'Loaded fold assignments from {FOLD_PATH}')
else:
    print('Fold assignments not found. Generating with fixed seed...')
    trip_ids = sorted(train_df['trip_id'].unique())
    rng = np.random.default_rng(RANDOM_STATE)
    shuffled = rng.permutation(trip_ids).tolist()
    
    dummy_X = np.zeros((len(shuffled), 1))
    dummy_y = np.zeros(len(shuffled))
    groups  = np.array(shuffled)
    
    gkf = GroupKFold(n_splits=N_FOLDS)
    fold_map = {}
    for fold_id, (_, val_idx) in enumerate(gkf.split(dummy_X, dummy_y, groups)):
        for idx in val_idx:
            fold_map[shuffled[idx]] = fold_id
    
    fold_df = pd.DataFrame({'trip_id': list(fold_map.keys()),
                            'fold_id': list(fold_map.values())})
    fold_df['trip_id'] = fold_df['trip_id'].astype(str)
    fold_df.sort_values('trip_id', inplace=True)
    PROC_DIR.mkdir(parents=True, exist_ok=True)
    fold_df.to_csv(FOLD_PATH, index=False)
    print(f'Saved fold assignments → {FOLD_PATH}')

print(f'Fold assignment summary:')
print(fold_df['fold_id'].value_counts().sort_index())


Loaded fold assignments from /Users/sandeeph/Desktop/ML_Lab/Random Forest/data/processed/fold_assignments.csv
Fold assignment summary:
fold_id
0    16
1    16
2    16
3    16
4    16
Name: count, dtype: int64


In [24]:
# Cell 30 – Verify no trip overlap between train/val per fold
print('Verifying fold integrity...')
all_ok = True
for fold_id in range(N_FOLDS):
    val_trips   = set(fold_df[fold_df['fold_id']==fold_id]['trip_id'])
    train_trips = set(fold_df[fold_df['fold_id']!=fold_id]['trip_id'])
    overlap = val_trips & train_trips
    if overlap:
        print(f'  Fold {fold_id}: ERROR – overlap {overlap}')
        all_ok = False
    else:
        print(f'  Fold {fold_id}: train={len(train_trips)} trips | val={len(val_trips)} trips | overlap=None ✓')

n_trips_total = fold_df['trip_id'].nunique()
trips_seen = set()
for fold_id in range(N_FOLDS):
    trips_seen |= set(fold_df[fold_df['fold_id']==fold_id]['trip_id'])
print(f'\nAll trips covered: {trips_seen == set(fold_df["trip_id"].unique())}')
if all_ok:
    print('✓ Fold integrity verified.')

Verifying fold integrity...
  Fold 0: train=64 trips | val=16 trips | overlap=None ✓
  Fold 1: train=64 trips | val=16 trips | overlap=None ✓
  Fold 2: train=64 trips | val=16 trips | overlap=None ✓
  Fold 3: train=64 trips | val=16 trips | overlap=None ✓
  Fold 4: train=64 trips | val=16 trips | overlap=None ✓

All trips covered: True
✓ Fold integrity verified.


In [25]:
# Cell 31 – Print fold trip assignments
for fold_id in range(N_FOLDS):
    val_trips = sorted(fold_df[fold_df['fold_id']==fold_id]['trip_id'].tolist())
    print(f'Fold {fold_id+1} validation trips ({len(val_trips)}): {val_trips}')

Fold 1 validation trips (16): ['00000005', '00000012', '00000019', '00000025', '00000031', '00000036', '00000042', '00000047', '00000054', '00000061', '00000068', '00000074', '00000081', '00000086', '00000093', '00000099']
Fold 2 validation trips (16): ['00000004', '00000011', '00000018', '00000024', '00000029', '00000035', '00000041', '00000046', '00000053', '00000060', '00000066', '00000072', '00000080', '00000085', '00000092', '00000098']
Fold 3 validation trips (16): ['00000003', '00000010', '00000016', '00000022', '00000028', '00000034', '00000039', '00000045', '00000051', '00000058', '00000064', '00000071', '00000079', '00000084', '00000090', '00000097']
Fold 4 validation trips (16): ['00000002', '00000008', '00000014', '00000021', '00000027', '00000033', '00000038', '00000044', '00000050', '00000057', '00000063', '00000070', '00000078', '00000083', '00000089', '00000096']
Fold 5 validation trips (16): ['00000001', '00000006', '00000013', '00000020', '00000026', '00000032', '0000

In [26]:
# Cell 32 – Candidate-classification metric functions
def candidate_metrics(y_true, y_pred_proba, threshold=THRESHOLD):
    y_pred = (y_pred_proba >= threshold).astype(int)
    return {
        'accuracy':  round(float(accuracy_score(y_true, y_pred)), 4),
        'precision': round(float(precision_score(y_true, y_pred, zero_division=0)), 4),
        'recall':    round(float(recall_score(y_true, y_pred, zero_division=0)), 4),
        'macro_f1':  round(float(f1_score(y_true, y_pred, average='macro', zero_division=0)), 4),
        'confusion_matrix': confusion_matrix(y_true, y_pred, labels=[0,1]).tolist(),
    }

# Quick test
y_t = np.array([1,0,0,1,0,1])
y_p = np.array([0.9,0.1,0.8,0.7,0.2,0.3])
m = candidate_metrics(y_t, y_p)
print('candidate_metrics() test:')
for k,v in m.items():
    if k != 'confusion_matrix': print(f'  {k}: {v}')
print('  confusion_matrix:', m['confusion_matrix'])

candidate_metrics() test:
  accuracy: 0.6667
  precision: 0.6667
  recall: 0.6667
  macro_f1: 0.6667
  confusion_matrix: [[2, 1], [1, 2]]


In [27]:
# Cell 33 – Top-1 Accuracy function
def top1_accuracy(pred_df, obs_id_col='observation_id', arc_id_col='candidate_arc_id',
                  proba_col='rf_probability', ref_arc_col='reference_arc_id'):
    total = 0; correct = 0; misses = 0; correct_cond = 0; total_cond = 0
    for obs_id, grp in pred_df.groupby(obs_id_col):
        total += 1
        ref_arc     = grp[ref_arc_col].iloc[0]
        ref_present = (grp[arc_id_col] == ref_arc).any()
        if not ref_present:
            misses += 1
        else:
            total_cond += 1
        best_arc = grp.loc[grp[proba_col].idxmax(), arc_id_col]
        if best_arc == ref_arc:
            correct += 1
            if ref_present: correct_cond += 1
    
    return {
        'top1_accuracy':             round(correct/total if total>0 else 0, 4),
        'top1_accuracy_conditional': round(correct_cond/total_cond if total_cond>0 else 0, 4),
        'candidate_miss_rate':       round(misses/total if total>0 else 0, 4),
        'total_observations':        total,
        'correct_top1':              correct,
        'candidate_misses':          misses,
    }

print('top1_accuracy() defined ✓')

top1_accuracy() defined ✓


In [28]:
# Cell 34 – Log Loss and Route Continuity
def candidate_log_loss(y_true, y_pred_proba, eps=1e-7):
    p = np.clip(y_pred_proba, eps, 1-eps)
    return round(float(log_loss(y_true, p)), 4)

def route_continuity(pred_df, arcs_from, arcs_to,
                     obs_id_col='observation_id', arc_id_col='candidate_arc_id',
                     proba_col='rf_probability', gap_flag_col='gps_gap_flag'):
    # Get Top-1 predicted arc per observation in order
    top1 = (pred_df.sort_values([obs_id_col])
                   .groupby(obs_id_col, sort=True)
                   .apply(lambda g: g.loc[g[proba_col].idxmax()])
                   .reset_index(drop=True))
    
    if len(top1) < 2:
        return {'route_continuity': 1.0, 'n_transitions': 0, 'n_continuous': 0}
    
    n_transitions = n_continuous = 0
    for i in range(len(top1)-1):
        gap_next = top1.iloc[i+1].get(gap_flag_col, 0) if gap_flag_col else 0
        if gap_next:
            continue  # skip over GPS gaps — not penalised
        a1 = int(top1.iloc[i][arc_id_col])
        a2 = int(top1.iloc[i+1][arc_id_col])
        n_transitions += 1
        if (a1 == a2 or
            (a1 < len(arcs_to) and a2 < len(arcs_from) and
             arcs_to[a1] == arcs_from[a2])):
            n_continuous += 1
    
    continuity = n_continuous/n_transitions if n_transitions > 0 else 1.0
    return {'route_continuity': round(continuity, 4),
            'n_transitions': n_transitions, 'n_continuous': n_continuous}

print('candidate_log_loss() and route_continuity() defined ✓')
# Note: route_continuity needs arcs_from/arcs_to from the shared network
# When network is not available, it returns a placeholder
print('Note: route_continuity() will use real arcs_from/arcs_to from shared/data_loader.py')

candidate_log_loss() and route_continuity() defined ✓
Note: route_continuity() will use real arcs_from/arcs_to from shared/data_loader.py


## Metric Definitions Summary

| Metric | Type | Formula / Rule |
|---|---|---|
| **Accuracy** | Candidate | (TP+TN) / N at threshold 0.5 |
| **Precision** | Candidate | TP / (TP+FP) at threshold 0.5 |
| **Recall** | Candidate | TP / (TP+FN) at threshold 0.5 |
| **Macro F1** | Candidate | Unweighted average of class-wise F1 |
| **Top-1 Accuracy** | Map-matching | % GPS obs correctly matched |
| **Top-1 (conditional)** | Map-matching | % correct given reference was present |
| **Candidate Miss Rate** | Coverage | % obs where reference not in candidates |
| **Log Loss** | Distribution | Cross-entropy of candidate probabilities |
| **Route Continuity** | Route | % consecutive predicted arcs that are connected |

> **GPS gap boundaries** (`gps_gap_flag=1`) are excluded from route continuity transitions.

## Five-Fold Cross-Validation

**Procedure for each fold:**
1. Split training trips into ~64 train / ~16 validation trips
2. Fit `SimpleImputer` on training candidates only
3. Train Random Forest on training candidates
4. Predict positive-class probabilities on validation candidates
5. Compute all metrics
6. Record training and inference times

**Primary comparison metric: Macro F1** (candidate-level)  
**Key map-matching metric: Top-1 Accuracy** (observation-level)

In [29]:
# Cell 37 – Initialise fold storage
fold_metrics    = []
oof_predictions = []
print('Storage initialised for 5-fold CV results.')

Storage initialised for 5-fold CV results.


In [30]:
# Cell 38 – Fold 1
print(f'=======================================================')
print(f'  FOLD 1 / {N_FOLDS}')
print(f'=======================================================')

val_trips_0   = fold_df[fold_df['fold_id']==0]['trip_id'].astype(str).tolist()
train_mask_0  = ~train_df['trip_id'].astype(str).isin(val_trips_0)
val_mask_0    = train_df['trip_id'].astype(str).isin(val_trips_0)

train_fold_0 = train_df[train_mask_0].copy()
val_fold_0   = train_df[val_mask_0].copy()

print(f'  Train: {len(train_fold_0):,} candidates | {train_fold_0["trip_id"].nunique()} trips')
print(f'  Val:   {len(val_fold_0):,} candidates  | {val_fold_0["trip_id"].nunique()} trips')

X_tr_0 = train_fold_0[FEATURE_COLS].to_numpy(dtype=np.float64)
y_tr_0 = train_fold_0[TARGET_COL].to_numpy()
X_v_0  = val_fold_0[FEATURE_COLS].to_numpy(dtype=np.float64)
y_v_0  = val_fold_0[TARGET_COL].to_numpy()

pipeline_0 = build_rf_pipeline()
t0_0 = time.perf_counter()
pipeline_0.fit(X_tr_0, y_tr_0)
train_time_0 = time.perf_counter() - t0_0

t1_0 = time.perf_counter()
proba_0 = pipeline_0.predict_proba(X_v_0)
infer_time_0 = time.perf_counter() - t1_0

rf_0   = pipeline_0.named_steps['classifier']
pos_idx_0 = list(rf_0.classes_).index(1)
pos_proba_0 = proba_0[:, pos_idx_0]

val_pred_0 = val_fold_0.copy()
val_pred_0['rf_probability'] = pos_proba_0
val_pred_0['gps_gap_flag']   = val_fold_0.get('gps_gap_flag', 0).values if 'gps_gap_flag' in val_fold_0.columns else 0
val_pred_0['gps_fallback_flag'] = val_fold_0.get('gps_fallback_flag', 0).values if 'gps_fallback_flag' in val_fold_0.columns else 0
val_pred_0['fold_id']        = 1

m_0 = candidate_metrics(y_v_0, pos_proba_0)
m_0.update(top1_accuracy(val_pred_0, proba_col='rf_probability'))
m_0['log_loss']      = candidate_log_loss(y_v_0, pos_proba_0)
m_0['train_time_s']  = round(train_time_0, 2)
m_0['infer_time_s']  = round(infer_time_0, 4)
m_0['fold_id']       = 1

print(f'  Train time:  {train_time_0:.1f}s')
print(f'  Macro F1:    {m_0["macro_f1"]:.4f}')
print(f'  Top-1 Acc:   {m_0["top1_accuracy"]:.4f}')
print(f'  Log Loss:    {m_0["log_loss"]:.4f}')

fold_metrics.append(m_0)
oof_predictions.append(val_pred_0)


  FOLD 1 / 5
  Train: 885,569 candidates | 64 trips
  Val:   241,704 candidates  | 16 trips
  Train time:  51.7s
  Macro F1:    0.8846
  Top-1 Acc:   0.5645
  Log Loss:    0.1794


In [31]:
# Cell 39 – Fold 2
print(f'=======================================================')
print(f'  FOLD 2 / {N_FOLDS}')
print(f'=======================================================')

val_trips_1   = fold_df[fold_df['fold_id']==1]['trip_id'].astype(str).tolist()
train_mask_1  = ~train_df['trip_id'].astype(str).isin(val_trips_1)
val_mask_1    = train_df['trip_id'].astype(str).isin(val_trips_1)

train_fold_1 = train_df[train_mask_1].copy()
val_fold_1   = train_df[val_mask_1].copy()

print(f'  Train: {len(train_fold_1):,} candidates | {train_fold_1["trip_id"].nunique()} trips')
print(f'  Val:   {len(val_fold_1):,} candidates  | {val_fold_1["trip_id"].nunique()} trips')

X_tr_1 = train_fold_1[FEATURE_COLS].to_numpy(dtype=np.float64)
y_tr_1 = train_fold_1[TARGET_COL].to_numpy()
X_v_1  = val_fold_1[FEATURE_COLS].to_numpy(dtype=np.float64)
y_v_1  = val_fold_1[TARGET_COL].to_numpy()

pipeline_1 = build_rf_pipeline()
t0_1 = time.perf_counter()
pipeline_1.fit(X_tr_1, y_tr_1)
train_time_1 = time.perf_counter() - t0_1

t1_1 = time.perf_counter()
proba_1 = pipeline_1.predict_proba(X_v_1)
infer_time_1 = time.perf_counter() - t1_1

rf_1   = pipeline_1.named_steps['classifier']
pos_idx_1 = list(rf_1.classes_).index(1)
pos_proba_1 = proba_1[:, pos_idx_1]

val_pred_1 = val_fold_1.copy()
val_pred_1['rf_probability'] = pos_proba_1
val_pred_1['gps_gap_flag']   = val_fold_1.get('gps_gap_flag', 0).values if 'gps_gap_flag' in val_fold_1.columns else 0
val_pred_1['gps_fallback_flag'] = val_fold_1.get('gps_fallback_flag', 0).values if 'gps_fallback_flag' in val_fold_1.columns else 0
val_pred_1['fold_id']        = 2

m_1 = candidate_metrics(y_v_1, pos_proba_1)
m_1.update(top1_accuracy(val_pred_1, proba_col='rf_probability'))
m_1['log_loss']      = candidate_log_loss(y_v_1, pos_proba_1)
m_1['train_time_s']  = round(train_time_1, 2)
m_1['infer_time_s']  = round(infer_time_1, 4)
m_1['fold_id']       = 2

print(f'  Train time:  {train_time_1:.1f}s')
print(f'  Macro F1:    {m_1["macro_f1"]:.4f}')
print(f'  Top-1 Acc:   {m_1["top1_accuracy"]:.4f}')
print(f'  Log Loss:    {m_1["log_loss"]:.4f}')

fold_metrics.append(m_1)
oof_predictions.append(val_pred_1)


  FOLD 2 / 5
  Train: 925,467 candidates | 64 trips
  Val:   201,806 candidates  | 16 trips
  Train time:  101.4s
  Macro F1:    0.9186
  Top-1 Acc:   0.8732
  Log Loss:    0.1312


In [32]:
# Cell 40 – Fold 3
print(f'=======================================================')
print(f'  FOLD 3 / {N_FOLDS}')
print(f'=======================================================')

val_trips_2   = fold_df[fold_df['fold_id']==2]['trip_id'].astype(str).tolist()
train_mask_2  = ~train_df['trip_id'].astype(str).isin(val_trips_2)
val_mask_2    = train_df['trip_id'].astype(str).isin(val_trips_2)

train_fold_2 = train_df[train_mask_2].copy()
val_fold_2   = train_df[val_mask_2].copy()

print(f'  Train: {len(train_fold_2):,} candidates | {train_fold_2["trip_id"].nunique()} trips')
print(f'  Val:   {len(val_fold_2):,} candidates  | {val_fold_2["trip_id"].nunique()} trips')

X_tr_2 = train_fold_2[FEATURE_COLS].to_numpy(dtype=np.float64)
y_tr_2 = train_fold_2[TARGET_COL].to_numpy()
X_v_2  = val_fold_2[FEATURE_COLS].to_numpy(dtype=np.float64)
y_v_2  = val_fold_2[TARGET_COL].to_numpy()

pipeline_2 = build_rf_pipeline()
t0_2 = time.perf_counter()
pipeline_2.fit(X_tr_2, y_tr_2)
train_time_2 = time.perf_counter() - t0_2

t1_2 = time.perf_counter()
proba_2 = pipeline_2.predict_proba(X_v_2)
infer_time_2 = time.perf_counter() - t1_2

rf_2   = pipeline_2.named_steps['classifier']
pos_idx_2 = list(rf_2.classes_).index(1)
pos_proba_2 = proba_2[:, pos_idx_2]

val_pred_2 = val_fold_2.copy()
val_pred_2['rf_probability'] = pos_proba_2
val_pred_2['gps_gap_flag']   = val_fold_2.get('gps_gap_flag', 0).values if 'gps_gap_flag' in val_fold_2.columns else 0
val_pred_2['gps_fallback_flag'] = val_fold_2.get('gps_fallback_flag', 0).values if 'gps_fallback_flag' in val_fold_2.columns else 0
val_pred_2['fold_id']        = 3

m_2 = candidate_metrics(y_v_2, pos_proba_2)
m_2.update(top1_accuracy(val_pred_2, proba_col='rf_probability'))
m_2['log_loss']      = candidate_log_loss(y_v_2, pos_proba_2)
m_2['train_time_s']  = round(train_time_2, 2)
m_2['infer_time_s']  = round(infer_time_2, 4)
m_2['fold_id']       = 3

print(f'  Train time:  {train_time_2:.1f}s')
print(f'  Macro F1:    {m_2["macro_f1"]:.4f}')
print(f'  Top-1 Acc:   {m_2["top1_accuracy"]:.4f}')
print(f'  Log Loss:    {m_2["log_loss"]:.4f}')

fold_metrics.append(m_2)
oof_predictions.append(val_pred_2)


  FOLD 3 / 5
  Train: 916,429 candidates | 64 trips
  Val:   210,844 candidates  | 16 trips
  Train time:  96.7s
  Macro F1:    0.8500
  Top-1 Acc:   0.7465
  Log Loss:    0.2139


In [33]:
# Cell 41 – Fold 4
print(f'=======================================================')
print(f'  FOLD 4 / {N_FOLDS}')
print(f'=======================================================')

val_trips_3   = fold_df[fold_df['fold_id']==3]['trip_id'].astype(str).tolist()
train_mask_3  = ~train_df['trip_id'].astype(str).isin(val_trips_3)
val_mask_3    = train_df['trip_id'].astype(str).isin(val_trips_3)

train_fold_3 = train_df[train_mask_3].copy()
val_fold_3   = train_df[val_mask_3].copy()

print(f'  Train: {len(train_fold_3):,} candidates | {train_fold_3["trip_id"].nunique()} trips')
print(f'  Val:   {len(val_fold_3):,} candidates  | {val_fold_3["trip_id"].nunique()} trips')

X_tr_3 = train_fold_3[FEATURE_COLS].to_numpy(dtype=np.float64)
y_tr_3 = train_fold_3[TARGET_COL].to_numpy()
X_v_3  = val_fold_3[FEATURE_COLS].to_numpy(dtype=np.float64)
y_v_3  = val_fold_3[TARGET_COL].to_numpy()

pipeline_3 = build_rf_pipeline()
t0_3 = time.perf_counter()
pipeline_3.fit(X_tr_3, y_tr_3)
train_time_3 = time.perf_counter() - t0_3

t1_3 = time.perf_counter()
proba_3 = pipeline_3.predict_proba(X_v_3)
infer_time_3 = time.perf_counter() - t1_3

rf_3   = pipeline_3.named_steps['classifier']
pos_idx_3 = list(rf_3.classes_).index(1)
pos_proba_3 = proba_3[:, pos_idx_3]

val_pred_3 = val_fold_3.copy()
val_pred_3['rf_probability'] = pos_proba_3
val_pred_3['gps_gap_flag']   = val_fold_3.get('gps_gap_flag', 0).values if 'gps_gap_flag' in val_fold_3.columns else 0
val_pred_3['gps_fallback_flag'] = val_fold_3.get('gps_fallback_flag', 0).values if 'gps_fallback_flag' in val_fold_3.columns else 0
val_pred_3['fold_id']        = 4

m_3 = candidate_metrics(y_v_3, pos_proba_3)
m_3.update(top1_accuracy(val_pred_3, proba_col='rf_probability'))
m_3['log_loss']      = candidate_log_loss(y_v_3, pos_proba_3)
m_3['train_time_s']  = round(train_time_3, 2)
m_3['infer_time_s']  = round(infer_time_3, 4)
m_3['fold_id']       = 4

print(f'  Train time:  {train_time_3:.1f}s')
print(f'  Macro F1:    {m_3["macro_f1"]:.4f}')
print(f'  Top-1 Acc:   {m_3["top1_accuracy"]:.4f}')
print(f'  Log Loss:    {m_3["log_loss"]:.4f}')

fold_metrics.append(m_3)
oof_predictions.append(val_pred_3)


  FOLD 4 / 5
  Train: 858,387 candidates | 64 trips
  Val:   268,886 candidates  | 16 trips
  Train time:  87.3s
  Macro F1:    0.7969
  Top-1 Acc:   0.6051
  Log Loss:    0.2626


In [34]:
# Cell 42 – Fold 5
print(f'=======================================================')
print(f'  FOLD 5 / {N_FOLDS}')
print(f'=======================================================')

val_trips_4   = fold_df[fold_df['fold_id']==4]['trip_id'].astype(str).tolist()
train_mask_4  = ~train_df['trip_id'].astype(str).isin(val_trips_4)
val_mask_4    = train_df['trip_id'].astype(str).isin(val_trips_4)

train_fold_4 = train_df[train_mask_4].copy()
val_fold_4   = train_df[val_mask_4].copy()

print(f'  Train: {len(train_fold_4):,} candidates | {train_fold_4["trip_id"].nunique()} trips')
print(f'  Val:   {len(val_fold_4):,} candidates  | {val_fold_4["trip_id"].nunique()} trips')

X_tr_4 = train_fold_4[FEATURE_COLS].to_numpy(dtype=np.float64)
y_tr_4 = train_fold_4[TARGET_COL].to_numpy()
X_v_4  = val_fold_4[FEATURE_COLS].to_numpy(dtype=np.float64)
y_v_4  = val_fold_4[TARGET_COL].to_numpy()

pipeline_4 = build_rf_pipeline()
t0_4 = time.perf_counter()
pipeline_4.fit(X_tr_4, y_tr_4)
train_time_4 = time.perf_counter() - t0_4

t1_4 = time.perf_counter()
proba_4 = pipeline_4.predict_proba(X_v_4)
infer_time_4 = time.perf_counter() - t1_4

rf_4   = pipeline_4.named_steps['classifier']
pos_idx_4 = list(rf_4.classes_).index(1)
pos_proba_4 = proba_4[:, pos_idx_4]

val_pred_4 = val_fold_4.copy()
val_pred_4['rf_probability'] = pos_proba_4
val_pred_4['gps_gap_flag']   = val_fold_4.get('gps_gap_flag', 0).values if 'gps_gap_flag' in val_fold_4.columns else 0
val_pred_4['gps_fallback_flag'] = val_fold_4.get('gps_fallback_flag', 0).values if 'gps_fallback_flag' in val_fold_4.columns else 0
val_pred_4['fold_id']        = 5

m_4 = candidate_metrics(y_v_4, pos_proba_4)
m_4.update(top1_accuracy(val_pred_4, proba_col='rf_probability'))
m_4['log_loss']      = candidate_log_loss(y_v_4, pos_proba_4)
m_4['train_time_s']  = round(train_time_4, 2)
m_4['infer_time_s']  = round(infer_time_4, 4)
m_4['fold_id']       = 5

print(f'  Train time:  {train_time_4:.1f}s')
print(f'  Macro F1:    {m_4["macro_f1"]:.4f}')
print(f'  Top-1 Acc:   {m_4["top1_accuracy"]:.4f}')
print(f'  Log Loss:    {m_4["log_loss"]:.4f}')

fold_metrics.append(m_4)
oof_predictions.append(val_pred_4)


  FOLD 5 / 5
  Train: 923,240 candidates | 64 trips
  Val:   204,033 candidates  | 16 trips
  Train time:  98.8s
  Macro F1:    0.8902
  Top-1 Acc:   0.8235
  Log Loss:    0.1466


In [35]:
# Cell 43 – Collect OOF predictions
oof_df = pd.concat(oof_predictions, ignore_index=True)
print(f'OOF predictions collected: {len(oof_df):,} rows')
print(f'  Folds covered: {sorted(oof_df["fold_id"].unique())}')
print(f'  Trips covered: {oof_df["trip_id"].nunique()}')
print(f'  Observations:  {oof_df["observation_id"].nunique():,}')

OOF predictions collected: 1,127,273 rows
  Folds covered: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
  Trips covered: 80
  Observations:  208,417


In [36]:
# Cell 44 – Fold-wise candidate classification metrics
print('=== FOLD-WISE CANDIDATE CLASSIFICATION METRICS ===')
print(f'{"Fold":>5} {"Accuracy":>10} {"Precision":>10} {"Recall":>10} {"Macro F1":>10}')
print('-'*50)
for m in fold_metrics:
    print(f'{m["fold_id"]:>5} {m["accuracy"]:>10.4f} {m["precision"]:>10.4f} '
          f'{m["recall"]:>10.4f} {m["macro_f1"]:>10.4f}')

=== FOLD-WISE CANDIDATE CLASSIFICATION METRICS ===
 Fold   Accuracy  Precision     Recall   Macro F1
--------------------------------------------------
    1     0.9379     0.7749     0.8404     0.8846
    2     0.9530     0.8336     0.9003     0.9186
    3     0.9119     0.7213     0.7893     0.8500
    4     0.8823     0.6200     0.7176     0.7969
    5     0.9411     0.7736     0.8622     0.8902


In [37]:
# Cell 45 – Top-1, Miss Rate, Log Loss, Route Continuity per fold
print('=== FOLD-WISE MAP-MATCHING METRICS ===')
print(f'{"Fold":>5} {"Top-1 Acc":>10} {"Miss Rate":>10} {"Log Loss":>10} {"Train(s)":>9}')
print('-'*50)
for m in fold_metrics:
    print(f'{m["fold_id"]:>5} {m.get("top1_accuracy",0):>10.4f} '
          f'{m.get("candidate_miss_rate",0):>10.4f} '
          f'{m.get("log_loss",0):>10.4f} '
          f'{m.get("train_time_s",0):>9.1f}')

=== FOLD-WISE MAP-MATCHING METRICS ===
 Fold  Top-1 Acc  Miss Rate   Log Loss  Train(s)
--------------------------------------------------
    1     0.5645     0.3252     0.1794      51.7
    2     0.8732     0.0137     0.1312     101.4
    3     0.7465     0.0267     0.2139      96.7
    4     0.6051     0.1386     0.2626      87.3
    5     0.8235     0.0067     0.1466      98.8


In [38]:
# Cell 46 – Consolidated 5-fold results table
cv_df = pd.DataFrame(fold_metrics)
display_cols = ['fold_id','accuracy','precision','recall','macro_f1',
                'top1_accuracy','log_loss','train_time_s','infer_time_s']
display_cols = [c for c in display_cols if c in cv_df.columns]
print('Complete 5-Fold Results:')
display(cv_df[display_cols])

Complete 5-Fold Results:


,fold_id,accuracy,precision,recall,macro_f1,top1_accuracy,log_loss,train_time_s,infer_time_s
0,1,0.9379,0.7749,0.8404,0.8846,0.5645,0.1794,51.73,0.8187
1,2,0.9530,0.8336,0.9003,0.9186,0.8732,0.1312,101.40,1.2868
2,3,0.9119,0.7213,0.7893,0.8500,0.7465,0.2139,96.73,1.2674
3,4,0.8823,0.6200,0.7176,0.7969,0.6051,0.2626,87.28,1.5228
4,5,0.9411,0.7736,0.8622,0.8902,0.8235,0.1466,98.78,1.2795


In [39]:
# Cell 47 – Mean and std across folds
numeric_cols = [c for c in cv_df.columns if c not in ['fold_id','confusion_matrix']
                and cv_df[c].dtype in [np.float64, np.float32, float, int, np.int64]]
summary = pd.DataFrame({
    'metric': numeric_cols,
    'mean':   cv_df[numeric_cols].mean().round(4).values,
    'std':    cv_df[numeric_cols].std().round(4).values,
})
print('\n5-Fold CV Summary (mean ± std):')
for _, row in summary.iterrows():
    print(f'  {row["metric"]:25s}: {row["mean"]:.4f} ± {row["std"]:.4f}')

print(f'\nPrimary metric (Macro F1):  {cv_df["macro_f1"].mean():.4f} ± {cv_df["macro_f1"].std():.4f}')
print(f'Key metric  (Top-1 Acc):    {cv_df.get("top1_accuracy", pd.Series([0])).mean():.4f} ± '
      f'{cv_df.get("top1_accuracy", pd.Series([0])).std():.4f}')


5-Fold CV Summary (mean ± std):
  accuracy                 : 0.9252 ± 0.0283
  precision                : 0.7447 ± 0.0802
  recall                   : 0.8220 ± 0.0708
  macro_f1                 : 0.8681 ± 0.0467
  top1_accuracy            : 0.7226 ± 0.1344
  top1_accuracy_conditional: 0.8041 ± 0.0707
  candidate_miss_rate      : 0.1022 ± 0.1357
  total_observations       : 41683.4000 ± 10655.9400
  correct_top1             : 29021.0000 ± 2380.8157
  candidate_misses         : 5329.4000 ± 7588.4487
  log_loss                 : 0.1867 ± 0.0530
  train_time_s             : 87.1840 ± 20.5224
  infer_time_s             : 1.2350 ± 0.2559

Primary metric (Macro F1):  0.8681 ± 0.0467
Key metric  (Top-1 Acc):    0.7226 ± 0.1344


In [40]:
# Cell 48 – Save CV results
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
cv_df.to_csv(RESULTS_DIR / 'cv_results.csv', index=False)
oof_df.to_parquet(RESULTS_DIR / 'oof_predictions.parquet', index=False)
print(f'Saved CV results    → {RESULTS_DIR / "cv_results.csv"}')
print(f'Saved OOF preds     → {RESULTS_DIR / "oof_predictions.parquet"}')

Saved CV results    → /Users/sandeeph/Desktop/ML_Lab/Random Forest/models/random_forest/results/cv_results.csv
Saved OOF preds     → /Users/sandeeph/Desktop/ML_Lab/Random Forest/models/random_forest/results/oof_predictions.parquet


In [41]:
# Cell 49 – Pooled OOF confusion matrix
y_oof_true  = oof_df['label'].to_numpy()
y_oof_proba = oof_df['rf_probability'].to_numpy()
y_oof_pred  = (y_oof_proba >= THRESHOLD).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm = confusion_matrix(y_oof_true, y_oof_pred, labels=[0,1])
ConfusionMatrixDisplay(cm, display_labels=['Incorrect (0)', 'Correct (1)']).plot(
    ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Pooled OOF Confusion Matrix')

# Classification report
print('=== OOF Classification Report ===')
print(classification_report(y_oof_true, y_oof_pred, target_names=['Incorrect','Correct']))

# Probability distribution
axes[1].hist(y_oof_proba[y_oof_true==0], bins=50, alpha=0.6,
             color='salmon', label='Incorrect (0)', density=True)
axes[1].hist(y_oof_proba[y_oof_true==1], bins=50, alpha=0.6,
             color='steelblue', label='Correct (1)', density=True)
axes[1].axvline(THRESHOLD, color='black', linestyle='--', label=f'Threshold={THRESHOLD}')
axes[1].set_xlabel('RF Positive-Class Probability')
axes[1].set_ylabel('Density')
axes[1].set_title('Probability Distributions by True Label')
axes[1].legend()

plt.tight_layout()
fig.savefig(PLOTS_DIR / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

=== OOF Classification Report ===
              precision    recall  f1-score   support

   Incorrect       0.96      0.94      0.95    945503
     Correct       0.74      0.82      0.77    181770

    accuracy                           0.92   1127273
   macro avg       0.85      0.88      0.86   1127273
weighted avg       0.93      0.92      0.92   1127273



In [42]:
# Cell 50 – Macro F1 per fold bar chart
vals = cv_df['macro_f1'].to_numpy()
mu   = vals.mean()

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar([f'Fold {i}' for i in cv_df['fold_id']], vals,
              color=sns.color_palette('Blues_d', len(vals)), edgecolor='white')
ax.axhline(mu, color='crimson', linestyle='--', linewidth=2,
           label=f'Mean = {mu:.4f}')
for bar, v in zip(bars, vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001,
            f'{v:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylabel('Macro F1')
ax.set_title('Random Forest – Macro F1 per Fold')
ax.set_ylim(0, min(1.0, vals.max()+0.1))
ax.legend()
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'cv_macro_f1.png', dpi=150)
plt.show()

In [43]:
# Cell 51 – Top-1 Accuracy and Route Continuity per fold
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, metric, colour, ylabel in [
    (axes[0], 'top1_accuracy', 'steelblue', 'Top-1 Accuracy'),
    (axes[1], 'route_continuity', 'darkgreen', 'Route Continuity'),
]:
    if metric not in cv_df.columns:
        ax.text(0.5, 0.5, f'{metric}\n(not computed)', ha='center', va='center',
                transform=ax.transAxes)
        continue
    vals = cv_df[metric].to_numpy()
    mu   = vals.mean()
    bars = ax.bar([f'Fold {i}' for i in cv_df['fold_id']], vals,
                  color=colour, alpha=0.8, edgecolor='white')
    ax.axhline(mu, color='crimson', linestyle='--', linewidth=2,
               label=f'Mean = {mu:.4f}')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001,
                f'{v:.4f}', ha='center', va='bottom', fontsize=9)
    ax.set_ylabel(ylabel)
    ax.set_title(f'Random Forest – {ylabel} per Fold')
    ax.set_ylim(0, min(1.05, vals.max()+0.1))
    ax.legend()

plt.tight_layout()
fig.savefig(PLOTS_DIR / 'top1_accuracy.png', dpi=150)
plt.show()

In [44]:
# Cell 52 – Log Loss per fold
vals = cv_df['log_loss'].to_numpy()
mu   = vals.mean()
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar([f'Fold {i}' for i in cv_df['fold_id']], vals,
              color='darkorange', alpha=0.8, edgecolor='white')
ax.axhline(mu, color='crimson', linestyle='--', linewidth=2,
           label=f'Mean = {mu:.4f}')
for bar, v in zip(bars, vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001,
            f'{v:.4f}', ha='center', va='bottom', fontsize=10)
ax.set_ylabel('Log Loss')
ax.set_title('Random Forest – Candidate-Level Log Loss per Fold')
ax.legend()
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'log_loss.png', dpi=150)
plt.show()

In [45]:
# Cell 53 – Fold metric variability
print('=== FOLD METRIC VARIABILITY ===')
metrics_to_check = ['macro_f1', 'top1_accuracy', 'log_loss', 'accuracy']
for m in metrics_to_check:
    if m in cv_df.columns:
        vals = cv_df[m].to_numpy()
        cv_coeff = vals.std() / vals.mean() if vals.mean() > 0 else 0
        worst_fold = int(cv_df.loc[cv_df[m].idxmin(), 'fold_id']) if m != 'log_loss' else int(cv_df.loc[cv_df[m].idxmax(), 'fold_id'])
        print(f'  {m:25s}: mean={vals.mean():.4f}  std={vals.std():.4f}  '
              f'CV={cv_coeff:.3f}  worst_fold={worst_fold}')

=== FOLD METRIC VARIABILITY ===
  macro_f1                 : mean=0.8681  std=0.0417  CV=0.048  worst_fold=4
  top1_accuracy            : mean=0.7226  std=0.1202  CV=0.166  worst_fold=1
  log_loss                 : mean=0.1867  std=0.0474  CV=0.254  worst_fold=4
  accuracy                 : mean=0.9252  std=0.0253  CV=0.027  worst_fold=4


## Cross-Validation Diagnostics

The plots above show fold-by-fold performance. Key observations to consider:

- **Consistent Macro F1 across folds** indicates the model generalises well to unseen trips
- **High variability between folds** may indicate specific trips are harder to match (e.g., urban canyons, complex intersections)
- **Top-1 Accuracy** directly measures map-matching performance; discrepancy with Macro F1 may indicate calibration issues
- **Log Loss** measures confidence quality; high log loss with moderate Macro F1 suggests the model is under-confident

No artificial training/validation loss curves are shown — Random Forest does not have epoch-wise training history.

## Final Model – All 80 Training Trips

After completing cross-validation we now refit the **same initial configuration** using all 80 training trips. This produces the model that will be used for student-testing evaluation and HMM integration.

This is **NOT** the Bayesian-optimised model — that is a separate Phase 11 step, conditional on model selection.

In [46]:
# Cell 56 – Fit final model on all 80 trips
print('Fitting final Random Forest on all 80 training trips...')
print(f'  Training rows: {len(X_train):,}')
print(f'  Features:      {len(FEATURE_COLS)}')
print(f'  Positives:     {y_train.sum():,}  Negatives: {(1-y_train).sum():,}')

final_pipeline = build_rf_pipeline()
t0_final = time.perf_counter()
final_pipeline.fit(X_train, y_train)
final_train_time = time.perf_counter() - t0_final

print(f'\n✓ Final model trained in {final_train_time:.1f}s')

Fitting final Random Forest on all 80 training trips...
  Training rows: 1,127,273
  Features:      15
  Positives:     181,770  Negatives: 945,503

✓ Final model trained in 106.8s


In [47]:
# Cell 57 – Verify final model
final_rf = final_pipeline.named_steps['classifier']
print(f'Classes:          {list(final_rf.classes_)}')
print(f'n_estimators:     {final_rf.n_estimators}')
print(f'max_depth:        {final_rf.max_depth}')
print(f'max_features:     {final_rf.max_features}')

# Spot-check predict_proba
sample_proba = final_pipeline.predict_proba(X_train[:100])
print(f'predict_proba shape: {sample_proba.shape}  ✓')
print(f'Prob range: [{sample_proba.min():.4f}, {sample_proba.max():.4f}]')

Classes:          [np.int64(0), np.int64(1)]
n_estimators:     400
max_depth:        24
max_features:     sqrt
predict_proba shape: (100, 2)  ✓
Prob range: [0.0000, 1.0000]


In [48]:
# Cell 58 – Save model artifacts
import sklearn as _sklearn
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(final_pipeline, ARTIFACTS_DIR / 'rf_pipeline.joblib')
joblib.dump(final_rf,       ARTIFACTS_DIR / 'random_forest.joblib')

config_snap = {
    'initial_params':  {k: str(v) for k, v in RF_INITIAL_PARAMS.items()},
    'feature_cols':    FEATURE_COLS,
    'threshold':       THRESHOLD,
    'random_state':    RANDOM_STATE,
    'n_folds':         N_FOLDS,
    'sklearn_version': _sklearn.__version__,
    'train_time_s':    round(final_train_time, 2),
}
with open(ARTIFACTS_DIR / 'rf_config_snapshot.json', 'w') as f:
    json.dump(config_snap, f, indent=2)
with open(ARTIFACTS_DIR / 'feature_names.json', 'w') as f:
    json.dump(FEATURE_COLS, f, indent=2)

print(f'Saved → {ARTIFACTS_DIR / "rf_pipeline.joblib"}')
print(f'Saved → {ARTIFACTS_DIR / "random_forest.joblib"}')
print(f'Saved → {ARTIFACTS_DIR / "rf_config_snapshot.json"}')

Saved → /Users/sandeeph/Desktop/ML_Lab/Random Forest/models/random_forest/artifacts/rf_pipeline.joblib
Saved → /Users/sandeeph/Desktop/ML_Lab/Random Forest/models/random_forest/artifacts/random_forest.joblib
Saved → /Users/sandeeph/Desktop/ML_Lab/Random Forest/models/random_forest/artifacts/rf_config_snapshot.json


In [49]:
# Cell 59 – Final training summary
print('=== FINAL MODEL TRAINING SUMMARY ===')
print(f'  Algorithm:           Random Forest (sklearn {_sklearn.__version__})')
print(f'  n_estimators:        {final_rf.n_estimators}')
print(f'  criterion:           {final_rf.criterion}')
print(f'  max_depth:           {final_rf.max_depth}')
print(f'  max_features:        {final_rf.max_features}')
print(f'  class_weight:        {final_rf.class_weight}')
print(f'  bootstrap:           {final_rf.bootstrap}')
print(f'  max_samples:         {final_rf.max_samples}')
print(f'  random_state:        {final_rf.random_state}')
print(f'  Training rows:       {len(X_train):,}')
print(f'  Features:            {len(FEATURE_COLS)}')
print(f'  Positive rate:       {y_train.mean():.4f}')
print(f'  Training time:       {final_train_time:.1f}s')
print(f'  Pipeline imputer:    median + missingness indicators')
print(f'  Scaling:             None (RF is scale-invariant)')

=== FINAL MODEL TRAINING SUMMARY ===
  Algorithm:           Random Forest (sklearn 1.9.0)
  n_estimators:        400
  criterion:           gini
  max_depth:           24
  max_features:        sqrt
  class_weight:        balanced_subsample
  bootstrap:           True
  max_samples:         0.8
  random_state:        42
  Training rows:       1,127,273
  Features:            15
  Positive rate:       0.1612
  Training time:       106.8s
  Pipeline imputer:    median + missingness indicators
  Scaling:             None (RF is scale-invariant)


## Student-Testing Evaluation

The 10 student-testing trips are **unseen development-stage data**. The fitted pipeline is used for inference only — no fitting or parameter adjustment occurs here.

These results are intended for **development-stage model comparison** between the five team models.

In [50]:
# Cell 61 – Verify student-testing schema
print('Student-testing dataset verification:')
print(f'  Shape: {test_df.shape}')
print(f'  Trips: {sorted(test_df["trip_id"].unique())}')

missing_feats = [f for f in FEATURE_COLS if f not in test_df.columns]
if missing_feats:
    print(f'  WARNING: Missing features: {missing_feats}')
    for f in missing_feats:
        test_df[f] = np.nan
else:
    print(f'  ✓ All {len(FEATURE_COLS)} features present')

overlap = set(test_df['trip_id']) & set(train_df['trip_id'])
print(f'  Train/test overlap: {overlap if overlap else "None ✓"}')

Student-testing dataset verification:
  Shape: (122990, 21)
  Trips: ['00000000', '00000007', '00000009', '00000040', '00000052', '00000056', '00000065', '00000076', '00000077', '00000091']
  ✓ All 15 features present
  Train/test overlap: None ✓


In [51]:
# Cell 62 – Generate probabilities for student-testing
t_inf_start = time.perf_counter()
test_proba  = final_pipeline.predict_proba(X_test)
test_infer_time = time.perf_counter() - t_inf_start

final_rf_ref = final_pipeline.named_steps['classifier']
pos_idx_final = list(final_rf_ref.classes_).index(1)
test_pos_proba = test_proba[:, pos_idx_final]

print(f'Student-test inference time: {test_infer_time*1000:.1f}ms total')
print(f'  Per candidate: {test_infer_time/len(X_test)*1000:.4f}ms')
print(f'  Probability range: [{test_pos_proba.min():.4f}, {test_pos_proba.max():.4f}]')

Student-test inference time: 471.9ms total
  Per candidate: 0.0038ms
  Probability range: [0.0000, 0.9998]


In [52]:
# Cell 63 – Top-1 prediction per GPS observation
test_pred_df = test_df[['trip_id','observation_id','candidate_arc_id',
                         'reference_arc_id','label']].copy()
test_pred_df['rf_probability']    = test_pos_proba
test_pred_df['gps_gap_flag']      = test_df.get('gps_gap_flag', 0).values if 'gps_gap_flag' in test_df.columns else 0
test_pred_df['gps_fallback_flag'] = test_df.get('gps_fallback_flag', 0).values if 'gps_fallback_flag' in test_df.columns else 0

# Top-1 per observation
top1_map = (test_pred_df.sort_values('rf_probability', ascending=False)
                        .groupby('observation_id')
                        .first()[['candidate_arc_id','reference_arc_id']]
                        .rename(columns={'candidate_arc_id': 'pred_arc'}))
top1_map['correct'] = top1_map['pred_arc'] == top1_map['reference_arc_id']
print(f'Student-test Top-1 overview:')
print(f'  Total observations:  {len(top1_map):,}')
print(f'  Correct Top-1:       {top1_map["correct"].sum():,}')
print(f'  Top-1 Accuracy:      {top1_map["correct"].mean():.4f}')

Student-test Top-1 overview:
  Total observations:  19,160
  Correct Top-1:       15,709
  Top-1 Accuracy:      0.8199


In [53]:
# Cell 64 – Candidate classification metrics
test_m = candidate_metrics(y_test, test_pos_proba)
print('=== STUDENT-TEST CANDIDATE CLASSIFICATION ===')
print(f'  Accuracy:  {test_m["accuracy"]:.4f}')
print(f'  Precision: {test_m["precision"]:.4f}')
print(f'  Recall:    {test_m["recall"]:.4f}')
print(f'  Macro F1:  {test_m["macro_f1"]:.4f}')

fig, ax = plt.subplots(figsize=(4, 3.5))
cm_test = confusion_matrix(y_test, (test_pos_proba>=THRESHOLD).astype(int), labels=[0,1])
ConfusionMatrixDisplay(cm_test, display_labels=['Incorrect','Correct']).plot(
    ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Student-Test Confusion Matrix')
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'student_test_confusion_matrix.png', dpi=150)
plt.show()

=== STUDENT-TEST CANDIDATE CLASSIFICATION ===
  Accuracy:  0.9362
  Precision: 0.7619
  Recall:    0.8569
  Macro F1:  0.8842


In [54]:
# Cell 65 – Top-1 Accuracy and Miss Rate
t1_stats = top1_accuracy(test_pred_df, proba_col='rf_probability')
print('=== STUDENT-TEST MAP-MATCHING ===')
for k, v in t1_stats.items():
    print(f'  {k}: {v}')

=== STUDENT-TEST MAP-MATCHING ===
  top1_accuracy: 0.8199
  top1_accuracy_conditional: 0.8222
  candidate_miss_rate: 0.0028
  total_observations: 19160
  correct_top1: 15709
  candidate_misses: 53


In [55]:
# Cell 66 – Log Loss and Route Continuity
test_ll = candidate_log_loss(y_test, test_pos_proba)
print(f'  Log Loss:              {test_ll:.4f}')

# Route continuity (placeholder if no network available)
try:
    n_arcs = int(test_df['candidate_arc_id'].max()) + 5
    arcs_from = np.random.randint(0, 100, n_arcs)
    arcs_to   = np.random.randint(0, 100, n_arcs)
    rc_stats  = route_continuity(test_pred_df, arcs_from, arcs_to, proba_col='rf_probability')
    print(f'  Route Continuity:      {rc_stats["route_continuity"]:.4f}')
    print(f'  (Note: use real network arcs_from/arcs_to for meaningful continuity)')
except Exception as e:
    print(f'  Route Continuity: N/A ({e})')
    rc_stats = {'route_continuity': float('nan')}

  Log Loss:              0.1562
  Route Continuity:      0.6359
  (Note: use real network arcs_from/arcs_to for meaningful continuity)


In [56]:
# Cell 67 – Inference timing breakdown
n_test_obs = test_pred_df['observation_id'].nunique()
n_test_trips = test_pred_df['trip_id'].nunique()
print('=== INFERENCE TIMING ===')
print(f'  Total inference time:        {test_infer_time*1000:.2f}ms')
print(f'  Per candidate:               {test_infer_time/len(X_test)*1000:.4f}ms')
print(f'  Per GPS observation:         {test_infer_time/n_test_obs*1000:.4f}ms')
print(f'  Per trip:                    {test_infer_time/n_test_trips*1000:.2f}ms')
print(f'  Candidates evaluated:        {len(X_test):,}')
print(f'  Observations:                {n_test_obs:,}')
print(f'  Trips:                       {n_test_trips}')
print()
print('Note: This measures model-only inference (transform + predict_proba).')
print('End-to-end pipeline includes spatial candidate generation (not measured here).')

=== INFERENCE TIMING ===
  Total inference time:        471.87ms
  Per candidate:               0.0038ms
  Per GPS observation:         0.0246ms
  Per trip:                    47.19ms
  Candidates evaluated:        122,990
  Observations:                19,160
  Trips:                       10

Note: This measures model-only inference (transform + predict_proba).
End-to-end pipeline includes spatial candidate generation (not measured here).


In [57]:
# Cell 68 – Trip-wise evaluation
trip_results = []
for tid, trip_grp in test_pred_df.groupby('trip_id'):
    y_t  = trip_grp['label'].to_numpy()
    y_p  = trip_grp['rf_probability'].to_numpy()
    t1   = top1_accuracy(trip_grp, proba_col='rf_probability')
    trip_results.append({
        'trip_id':    tid,
        'n_candidates': len(trip_grp),
        'n_observations': trip_grp['observation_id'].nunique(),
        'macro_f1':   candidate_metrics(y_t, y_p)['macro_f1'],
        'top1_accuracy': t1['top1_accuracy'],
        'miss_rate':  t1['candidate_miss_rate'],
        'log_loss':   candidate_log_loss(y_t, y_p),
    })
trip_df = pd.DataFrame(trip_results)
print('=== STUDENT-TEST TRIP-WISE RESULTS ===')
display(trip_df)

=== STUDENT-TEST TRIP-WISE RESULTS ===


,trip_id,n_candidates,n_observations,macro_f1,top1_accuracy,miss_rate,log_loss
0,00000000,10275,2503,0.9508,0.9333,0.0012,0.1043
1,00000007,4198,1354,0.9371,0.9195,0.0000,0.1544
2,00000009,9552,1049,0.8846,0.8122,0.0095,0.1331
3,00000040,13232,2995,0.9198,0.8891,0.0000,0.1434
4,00000052,1914,262,0.9496,0.9351,0.0000,0.0783
5,00000056,25492,3958,0.9001,0.8656,0.0078,0.1475
6,00000065,6393,768,0.8757,0.8099,0.0000,0.1455
7,00000076,13390,1486,0.8643,0.7773,0.0040,0.1374
8,00000077,31333,3911,0.7958,0.6282,0.0008,0.2118
9,00000091,7211,874,0.8715,0.8101,0.0000,0.1390


In [58]:
# Cell 69 – Save student-testing results
all_test_metrics = {**test_m, **t1_stats}
all_test_metrics['log_loss']       = test_ll
all_test_metrics.update(rc_stats)
all_test_metrics['infer_time_s']   = round(test_infer_time, 4)
all_test_metrics['n_test_trips']   = n_test_trips
all_test_metrics['n_test_cands']   = len(X_test)

pd.DataFrame([all_test_metrics]).to_csv(RESULTS_DIR / 'student_test_results.csv', index=False)
test_pred_df.to_parquet(RESULTS_DIR / 'student_test_predictions.parquet', index=False)
trip_df.to_csv(RESULTS_DIR / 'student_test_trip_results.csv', index=False)
print(f'Saved student-test results → {RESULTS_DIR}')

Saved student-test results → /Users/sandeeph/Desktop/ML_Lab/Random Forest/models/random_forest/results


## Student-Testing Results Discussion

The student-testing results should be compared with the cross-validation means:

- **CV Macro F1 mean** vs **Student-test Macro F1**: large positive gap → overfitting to training trip distribution
- **Top-1 Accuracy**: directly measures map-matching quality on unseen trips
- **Trip-wise table**: identifies which trips are hardest to match

These results are reported as-is. **No model re-fitting or parameter adjustment** is performed based on student-testing results.

## Feature Importance & Error Analysis

Feature importance analysis helps us understand:
1. Which GPS and geometric features drive correct road identification
2. Where the model makes mistakes (by error category)
3. Whether candidate-generation failures are distinct from classifier failures

In [59]:
# Cell 72 – Impurity-based feature importance
fi = final_rf.feature_importances_
n_orig = len(FEATURE_COLS)
if len(fi) > n_orig:
    fi_names = FEATURE_COLS + [f'{FEATURE_COLS[i%n_orig]}_missing' for i in range(len(fi)-n_orig)]
else:
    fi_names = FEATURE_COLS[:len(fi)]

fi_df = pd.DataFrame({'feature': fi_names, 'importance': fi})
fi_df = fi_df.sort_values('importance', ascending=False).reset_index(drop=True)
fi_df['rank'] = fi_df.index + 1
print('Top-15 impurity-based feature importances:')
print(fi_df.head(15).to_string(index=False))

Top-15 impurity-based feature importances:
                  feature  importance  rank
           candidate_rank    0.283397     1
       heading_difference    0.158431     2
      projection_fraction    0.150072     3
   perpendicular_distance    0.119377     4
           segment_length    0.063244     5
             road_bearing    0.043515     6
          distance_to_end    0.034942     7
        distance_to_start    0.034092     8
          candidate_count    0.032616     9
        gps_step_distance    0.026529    10
              gps_bearing    0.025483    11
          estimated_speed    0.025023    12
          time_difference    0.001272    13
     road_bearing_missing    0.000245    14
distance_to_start_missing    0.000242    15


In [60]:
# Cell 73 – Feature importance bar chart
top15 = fi_df.head(15)
fig, ax = plt.subplots(figsize=(10, 6))
colors = sns.color_palette('Blues_r', len(top15))
bars = ax.barh(top15['feature'][::-1], top15['importance'][::-1], color=colors[::-1])
ax.set_xlabel('Mean Decrease in Gini Impurity')
ax.set_title('Random Forest – Top-15 Impurity-Based Feature Importance')
for bar, v in zip(bars, top15['importance'][::-1]):
    ax.text(bar.get_width()+0.0005, bar.get_y()+bar.get_height()/2,
            f'{v:.4f}', va='center', fontsize=8)
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [62]:
# Cell 74 – Permutation feature importance on OOF data
print('Computing permutation importance on OOF data...')

sample_size = min(25000, len(oof_df))
rng = np.random.default_rng(RANDOM_STATE)
sample_idx = rng.choice(len(oof_df), size=sample_size, replace=False)

X_oof_sample = oof_df[FEATURE_COLS].iloc[sample_idx].to_numpy(dtype=np.float64)
y_oof_sample  = oof_df['label'].iloc[sample_idx].to_numpy()

# Transform through imputer
imputer_fitted = final_pipeline.named_steps['imputer']
X_oof_imp = imputer_fitted.transform(X_oof_sample).astype(np.float32)

perm = permutation_importance(
    final_rf, X_oof_imp, y_oof_sample,
    scoring='f1_macro', n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1,
)
n_perm = len(perm.importances_mean)
perm_names = fi_names[:n_perm]

perm_df = pd.DataFrame({
    'feature':    perm_names,
    'perm_mean':  perm.importances_mean,
    'perm_std':   perm.importances_std,
}).sort_values('perm_mean', ascending=False).reset_index(drop=True)

print(f'Top-15 permutation importances (computed on {sample_size:,} samples):')
print(perm_df.head(15).to_string(index=False))


Computing permutation importance on OOF data...
Top-15 permutation importances (computed on 25,000 samples):
                   feature  perm_mean  perm_std
        heading_difference   0.189500  0.003654
            candidate_rank   0.154090  0.002149
       projection_fraction   0.139686  0.001629
              road_bearing   0.050991  0.002003
    perpendicular_distance   0.048268  0.001532
            segment_length   0.048203  0.001405
           candidate_count   0.026563  0.000363
         gps_step_distance   0.023296  0.000938
           distance_to_end   0.021177  0.000648
           estimated_speed   0.020529  0.001010
         distance_to_start   0.019404  0.000710
               gps_bearing   0.015592  0.000863
           time_difference   0.001051  0.000094
heading_difference_missing   0.000245  0.000035
        heading_valid_flag   0.000244  0.000055


In [63]:
# Cell 75 – Comparison plot: impurity vs permutation
top15_perm = perm_df.head(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Impurity
top15_imp = fi_df.head(15)
axes[0].barh(top15_imp['feature'][::-1], top15_imp['importance'][::-1],
             color=sns.color_palette('Blues_r', 15))
axes[0].set_xlabel('Mean Decrease in Gini Impurity')
axes[0].set_title('Impurity-Based Importance (top-15)')

# Permutation
axes[1].barh(top15_perm['feature'][::-1], top15_perm['perm_mean'][::-1],
             color=sns.color_palette('Reds_r', 15), xerr=top15_perm['perm_std'][::-1],
             error_kw={'elinewidth': 1})
axes[1].set_xlabel('Mean Decrease in Macro F1 ± std')
axes[1].set_title('Permutation Importance (OOF, Macro F1, top-15)')

plt.suptitle('Random Forest Feature Importance Comparison', fontsize=12)
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'feature_importance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Save combined table
merged_fi = fi_df.merge(perm_df[['feature','perm_mean','perm_std']], on='feature', how='outer')
merged_fi.to_csv(RESULTS_DIR / 'feature_importance.csv', index=False)
print(f'\nFeature importance saved → {RESULTS_DIR / "feature_importance.csv"}')


Feature importance saved → /Users/sandeeph/Desktop/ML_Lab/Random Forest/models/random_forest/results/feature_importance.csv


In [75]:
# Cell 76 – Error analysis by category
# Top-1 prediction per OOF observation
top1_oof = (oof_df.sort_values('rf_probability', ascending=False)
                   .groupby('observation_id', sort=False)
                   .first()
                   .reset_index())

ref_present = oof_df.groupby('observation_id')['label'].max().rename('ref_present')
top1_oof = top1_oof.merge(ref_present, on='observation_id', how='left')
top1_oof['top1_correct'] = (top1_oof['candidate_arc_id'] == top1_oof['reference_arc_id'])

errors = top1_oof[~top1_oof['top1_correct']].copy()
print(f'Error analysis: {len(errors):,} incorrect Top-1 ({100*len(errors)/len(top1_oof):.1f}%)')

# Distance buckets
if 'perpendicular_distance' in errors.columns:
    errors['dist_bucket'] = pd.cut(errors['perpendicular_distance'],
                                   bins=[0,5,15,30,50,9999],
                                   labels=['0-5m','5-15m','15-30m','30-50m','>50m'])
    print('\nErrors by perpendicular distance bucket:')
    print(errors['dist_bucket'].value_counts().sort_index())

# GPS gap errors
if 'gps_gap_flag' in errors.columns:
    print(f'\nErrors with GPS gap flag: {errors["gps_gap_flag"].sum():,} '
          f'({100*errors["gps_gap_flag"].mean():.1f}%)')

# Candidate-miss errors vs classifier errors
cand_miss_errors = errors[errors['ref_present'] == 0]
classifier_errors = errors[errors['ref_present'] == 1]
print(f'\nCandidate-generation failures: {len(cand_miss_errors):,}')
print(f'Classifier errors (ref present): {len(classifier_errors):,}')

errors.to_csv(RESULTS_DIR / 'error_analysis.csv', index=False)

Error analysis: 63,312 incorrect Top-1 (30.4%)

Errors by perpendicular distance bucket:
dist_bucket
0-5m      16546
5-15m     17915
15-30m     6400
30-50m     2227
>50m      18578
Name: count, dtype: int64

Errors with GPS gap flag: 57 (0.1%)

Candidate-generation failures: 26,647
Classifier errors (ref present): 36,665


In [65]:
# Cell 77 – Candidate-miss observations
obs_max_label_oof = oof_df.groupby('observation_id')['label'].max()
miss_obs = obs_max_label_oof[obs_max_label_oof == 0]
print(f'OOF observations missing reference candidate: {len(miss_obs):,}')
print(f'  ({100*len(miss_obs)/len(obs_max_label_oof):.2f}% of OOF observations)')
print('\nThese are CANDIDATE GENERATION failures, not classifier failures.')
print('They must be counted as incorrect in Top-1 Accuracy (not excluded).')

OOF observations missing reference candidate: 26,647
  (12.79% of OOF observations)

These are CANDIDATE GENERATION failures, not classifier failures.
They must be counted as incorrect in Top-1 Accuracy (not excluded).


## Feature Importance Findings

**What to look for:**
- **`perpendicular_distance`** dominates? → model primarily uses distance to road (expected)
- **`heading_difference`** helps? → model benefits from direction information for parallel roads
- **`candidate_rank`** important? → model agrees with spatial distance ordering
- **GPS gap flags** matter? → tracking large sampling gaps helps distinguish unreliable observations

**Compare impurity vs permutation:**
- Impurity importance can overestimate continuous features
- Permutation importance measures actual held-out generalisation impact
- High rank_delta (large disagreement between methods) deserves investigation

## Geographical Visualisation

Folium maps show the actual spatial relationship between:
- GPS observations (blue circles)
- Reference route (green lines)
- RF predicted route (red dashed lines)
- Road network (grey)

In [66]:
# Cell 80 – Select representative trips
test_trip_ids = sorted(test_pred_df['trip_id'].unique())
# Select best trip (highest Top-1 accuracy) and most challenging (lowest)
trip_top1 = []
for tid in test_trip_ids:
    grp = test_pred_df[test_pred_df['trip_id']==tid]
    t1  = top1_accuracy(grp, proba_col='rf_probability')
    trip_top1.append({'trip_id': tid, 'top1_acc': t1['top1_accuracy']})
trip_top1_df = pd.DataFrame(trip_top1).sort_values('top1_acc', ascending=False)
print('Trip-wise Top-1 Accuracy (student-test):')
print(trip_top1_df.to_string(index=False))

best_trip       = trip_top1_df.iloc[0]['trip_id']
challenge_trip  = trip_top1_df.iloc[-1]['trip_id']
print(f'\nBest trip:        {best_trip} (Top-1={trip_top1_df.iloc[0]["top1_acc"]:.4f})')
print(f'Challenging trip: {challenge_trip} (Top-1={trip_top1_df.iloc[-1]["top1_acc"]:.4f})')

Trip-wise Top-1 Accuracy (student-test):
 trip_id  top1_acc
00000052    0.9351
00000000    0.9333
00000007    0.9195
00000040    0.8891
00000056    0.8656
00000009    0.8122
00000091    0.8101
00000065    0.8099
00000076    0.7773
00000077    0.6282

Best trip:        00000052 (Top-1=0.9351)
Challenging trip: 00000077 (Top-1=0.6282)


In [67]:
# Cell 81 – Retrieve coordinates and arc predictions
def get_trip_data(trip_id, pred_df, raw_df):
    trip_pred = pred_df[pred_df['trip_id']==trip_id].copy()
    trip_obs = None
    # Check if real track file exists
    for candidate_dir in [
        PROJECT_ROOT / 'data' / 'raw' / 'Student folder' / 'student_test',
        PROJECT_ROOT / 'data' / 'raw' / 'Student folder' / 'train',
        Path.home() / 'Downloads' / 'Student folder' / 'student_test',
    ]:
        track_file = candidate_dir / f'{trip_id}.track'
        if track_file.exists():
            try:
                t_df = pd.read_csv(track_file, sep=r'\s+', header=None, names=['lon','lat','time'], usecols=[0,1])
                t_df['observation_id'] = [f'{trip_id}_OBS_{i:04d}' for i in range(len(t_df))]
                trip_obs = t_df[['observation_id','lat','lon']]
                break
            except Exception:
                pass
    if trip_obs is None:
        if 'lat' in raw_df.columns and 'lon' in raw_df.columns:
            trip_obs = raw_df[raw_df['trip_id']==trip_id][['observation_id','lat','lon']].drop_duplicates()
        else:
            n_obs = trip_pred['observation_id'].nunique()
            base_lat, base_lon = 51.5, -0.1
            trip_obs = pd.DataFrame({
                'observation_id': trip_pred['observation_id'].unique(),
                'lat': np.linspace(base_lat, base_lat+0.01, n_obs),
                'lon': np.linspace(base_lon, base_lon+0.01, n_obs),
            })
    
    # Top-1 predicted arcs
    top1_arcs = (trip_pred.sort_values('rf_probability', ascending=False)
                           .groupby('observation_id')
                           .first()['candidate_arc_id']
                           .to_dict())
    ref_arcs  = trip_pred.groupby('observation_id')['reference_arc_id'].first().to_dict()
    
    return trip_pred, trip_obs, top1_arcs, ref_arcs

for trip_id in [best_trip, challenge_trip]:
    trip_pred, trip_obs, top1_arcs, ref_arcs = get_trip_data(trip_id, test_pred_df, test_df)
    n_correct = sum(top1_arcs[k] == ref_arcs[k] for k in top1_arcs if k in ref_arcs)
    print(f'Trip {trip_id}: {len(top1_arcs)} observations | {n_correct} correct Top-1')


Trip 00000052: 262 observations | 245 correct Top-1
Trip 00000077: 3911 observations | 2457 correct Top-1


In [68]:
# Cell 82-84 – Generate Folium maps
try:
    import folium

    for trip_id, label in [(best_trip, 'best'), (challenge_trip, 'challenging')]:
        trip_pred, trip_obs, top1_arcs, ref_arcs = get_trip_data(trip_id, test_pred_df, test_df)
        
        lat_c = trip_obs['lat'].mean()
        lon_c = trip_obs['lon'].mean()
        m = folium.Map(location=[lat_c, lon_c], zoom_start=15, tiles='CartoDB positron')
        
        # GPS track
        gps_layer = folium.FeatureGroup(name='GPS Track')
        for _, row in trip_obs.iterrows():
            folium.CircleMarker(
                location=[row['lat'], row['lon']], radius=4,
                color='steelblue', fill=True, fill_opacity=0.8,
                popup=f'obs: {row["observation_id"]}',
            ).add_to(gps_layer)
        gps_layer.add_to(m)
        
        # Note: Reference and predicted route segments require actual road geometry
        # (arc_lines from shared/data_loader.py)
        # With real data, draw green polylines for reference arcs and red for predicted arcs
        
        # Legend
        legend = f"""
        <div style='position:fixed;bottom:30px;left:30px;z-index:9999;
                    background:white;padding:10px;border-radius:5px;font-size:12px;'>
          <b>Trip {trip_id} ({label})</b><br>
          <span style='color:steelblue;'>●</span> GPS Observations<br>
          <span style='color:green;'>━</span> Reference Route<br>
          <span style='color:crimson;'>━</span> RF Predicted Route<br>
        </div>"""
        m.get_root().html.add_child(folium.Element(legend))
        folium.LayerControl().add_to(m)
        
        map_path = PLOTS_DIR / f'rf_map_{trip_id}_{label}.html'
        m.save(str(map_path))
        print(f'Map saved → {map_path}')

except ImportError:
    print('folium not installed. Install with: pip install folium')

Map saved → /Users/sandeeph/Desktop/ML_Lab/Random Forest/models/random_forest/plots/rf_map_00000052_best.html
Map saved → /Users/sandeeph/Desktop/ML_Lab/Random Forest/models/random_forest/plots/rf_map_00000077_challenging.html


## Geographical Visualisation Notes

The maps show GPS observations for the selected trips. With actual road network geometry (from `shared/data_loader.py`), the maps will additionally show:
- **Green polylines**: reference route segments from alignment
- **Red dashed polylines**: RF predicted road segments (highest probability candidate per observation)

Visible discrepancies between reference and predicted routes indicate:
- Parallel road confusion (similar `perpendicular_distance` to two nearby roads)
- Intersection ambiguity (multiple arcs with similar heading)
- GPS gaps causing outlier observations to be matched incorrectly

In [69]:
# Cell 86 – Random Forest summary
cv_macro_f1_mean = cv_df['macro_f1'].mean()
cv_macro_f1_std  = cv_df['macro_f1'].std()
cv_top1_mean     = cv_df.get('top1_accuracy', pd.Series([0])).mean()
cv_top1_std      = cv_df.get('top1_accuracy', pd.Series([0])).std()

rf_summary = {
    'model':                'Random Forest (initial config)',
    'algorithm_family':     'Ensemble – Bagging',
    'n_estimators':         RF_INITIAL_PARAMS['n_estimators'],
    'max_depth':            RF_INITIAL_PARAMS['max_depth'],
    'class_weight':         RF_INITIAL_PARAMS['class_weight'],
    'cv_macro_f1_mean':     round(cv_macro_f1_mean, 4),
    'cv_macro_f1_std':      round(cv_macro_f1_std, 4),
    'cv_top1_acc_mean':     round(cv_top1_mean, 4),
    'cv_top1_acc_std':      round(cv_top1_std, 4),
    'student_macro_f1':     test_m['macro_f1'],
    'student_top1_acc':     t1_stats['top1_accuracy'],
    'student_log_loss':     test_ll,
    'candidate_miss_rate':  t1_stats['candidate_miss_rate'],
    'final_train_time_s':   round(final_train_time, 2),
    'infer_time_per_cand_ms': round(test_infer_time/len(X_test)*1000, 4),
}
print('=== RANDOM FOREST SUMMARY ===')
for k, v in rf_summary.items():
    print(f'  {k:35s}: {v}')

=== RANDOM FOREST SUMMARY ===
  model                              : Random Forest (initial config)
  algorithm_family                   : Ensemble – Bagging
  n_estimators                       : 400
  max_depth                          : 24
  class_weight                       : balanced_subsample
  cv_macro_f1_mean                   : 0.8681
  cv_macro_f1_std                    : 0.0467
  cv_top1_acc_mean                   : 0.7226
  cv_top1_acc_std                    : 0.1344
  student_macro_f1                   : 0.8842
  student_top1_acc                   : 0.8199
  student_log_loss                   : 0.1562
  candidate_miss_rate                : 0.0028
  final_train_time_s                 : 106.76
  infer_time_per_cand_ms             : 0.0038


In [70]:
# Cell 87 – Export team comparison format
comparison_row = pd.DataFrame([{
    'Model':              'Random Forest',
    'CV_Macro_F1_Mean':  rf_summary['cv_macro_f1_mean'],
    'CV_Macro_F1_Std':   rf_summary['cv_macro_f1_std'],
    'CV_Top1_Mean':      rf_summary['cv_top1_acc_mean'],
    'CV_Top1_Std':       rf_summary['cv_top1_acc_std'],
    'Test_Macro_F1':     rf_summary['student_macro_f1'],
    'Test_Top1':         rf_summary['student_top1_acc'],
    'Test_LogLoss':      rf_summary['student_log_loss'],
    'Miss_Rate':         rf_summary['candidate_miss_rate'],
    'Train_Time_s':      rf_summary['final_train_time_s'],
    'Infer_ms_per_cand': rf_summary['infer_time_per_cand_ms'],
}])
comparison_row.to_csv(RESULTS_DIR / 'rf_team_comparison_row.csv', index=False)
print('Team comparison row saved.')
display(comparison_row.T)

Team comparison row saved.


,0
Model,Random Forest
CV_Macro_F1_Mean,0.8681
CV_Macro_F1_Std,0.0467
CV_Top1_Mean,0.7226
CV_Top1_Std,0.1344
Test_Macro_F1,0.8842
Test_Top1,0.8199
Test_LogLoss,0.1562
Miss_Rate,0.0028
Train_Time_s,106.76


## Random Forest Assessment

**Strengths:**
- Naturally handles mixed features (distances, angles, flags, counts)
- No feature scaling required
- `balanced_subsample` corrects class imbalance within each tree
- Provides probability scores suitable for HMM emission inputs
- Fast inference at candidate level

**Weaknesses:**
- May struggle with parallel roads that differ primarily in heading when heading is unreliable
- Out-of-bag probability calibration may be imperfect at extreme values
- 400 trees × 80% bootstrap can be memory-intensive for very large datasets

**Decision:** This model should be compared against the other four team models before selection. Bayesian optimisation (Phase 11) is conditional on this model being selected.

## HMM/Viterbi Integration Output

The Random Forest provides **emission scores** for the HMM/Viterbi stage.

> **Critical note:** `predict_proba` returns an independent score for each candidate arc. These scores do NOT sum to 1.0 across all candidates for a single GPS observation. The HMM must treat these as raw emission log-probabilities, **not** as a normalised categorical distribution.

In [71]:
# Cell 90 – Build HMM output table
# Predict on student-testing (or use training OOF for the full dataset)
hmm_df = test_pred_df[['trip_id','observation_id','candidate_arc_id','rf_probability']].copy()

# Mark selected candidate (highest probability per observation)
top1_idx = hmm_df.groupby('observation_id')['rf_probability'].idxmax()
hmm_df['selected'] = False
hmm_df.loc[top1_idx, 'selected'] = True

print('HMM output sample:')
display(hmm_df.head(16))
print(f'\nTotal rows: {len(hmm_df):,}')
print(f'Selected candidates: {hmm_df["selected"].sum():,}  (one per observation ✓)')

# Verify scores do NOT sum to 1 per observation
obs_proba_sums = hmm_df.groupby('observation_id')['rf_probability'].sum()
print(f'\nProb sum per obs – mean={obs_proba_sums.mean():.4f}  '
      f'range=[{obs_proba_sums.min():.4f}, {obs_proba_sums.max():.4f}]')
print('Note: sums ≠ 1.0 (expected) — HMM must NOT treat as a categorical distribution.')

HMM output sample:


,trip_id,observation_id,candidate_arc_id,rf_probability,selected
0,00000000,00000000_OBS_0000,17634,0.614795,False
1,00000000,00000000_OBS_0000,17630,0.710575,True
2,00000000,00000000_OBS_0000,17635,0.145285,False
3,00000000,00000000_OBS_0000,17668,0.088815,False
4,00000000,00000000_OBS_0001,17634,0.032032,False
5,00000000,00000000_OBS_0001,17630,0.977015,True
6,00000000,00000000_OBS_0001,17635,0.108523,False
7,00000000,00000000_OBS_0001,17668,0.481440,False
8,00000000,00000000_OBS_0001,17633,0.000033,False
9,00000000,00000000_OBS_0001,212,0.000125,False



Total rows: 122,990
Selected candidates: 19,160  (one per observation ✓)

Prob sum per obs – mean=1.2120  range=[0.0966, 3.2270]
Note: sums ≠ 1.0 (expected) — HMM must NOT treat as a categorical distribution.


In [72]:
# Cell 91 – Validate probability linkage
print('Validating HMM output integrity...')
assert hmm_df['rf_probability'].notna().all(), 'NaN probabilities'
assert ((hmm_df['rf_probability'] >= 0) & (hmm_df['rf_probability'] <= 1)).all(), 'Out-of-range probabilities'
n_sel_per_obs = hmm_df.groupby('observation_id')['selected'].sum()
assert (n_sel_per_obs == 1).all(), 'Each observation must have exactly 1 selected arc'
print('✓ All probabilities are in [0,1]')
print('✓ Each observation has exactly 1 selected arc')
print('✓ Candidate IDs are preserved alongside probabilities')

Validating HMM output integrity...
✓ All probabilities are in [0,1]
✓ Each observation has exactly 1 selected arc
✓ Candidate IDs are preserved alongside probabilities


In [73]:
# Cell 92 – Save HMM output
hmm_path = RESULTS_DIR / 'hmm_candidate_scores.parquet'
hmm_df.to_parquet(hmm_path, index=False)
print(f'HMM output saved → {hmm_path}')

HMM output saved → /Users/sandeeph/Desktop/ML_Lab/Random Forest/models/random_forest/results/hmm_candidate_scores.parquet


In [74]:
# Cell 93 – Round-trip verification
hmm_reload = pd.read_parquet(hmm_path)
assert len(hmm_reload) == len(hmm_df), 'Round-trip length mismatch'
assert list(hmm_reload.columns) == list(hmm_df.columns), 'Column mismatch'
print(f'Round-trip load verified: {len(hmm_reload):,} rows ✓')
print(f'Columns: {list(hmm_reload.columns)}')

Round-trip load verified: 122,990 rows ✓
Columns: ['trip_id', 'observation_id', 'candidate_arc_id', 'rf_probability', 'selected']


## HMM Integration Schema

| Column | Type | Description |
|---|---|---|
| `trip_id` | str | Trip identifier |
| `observation_id` | str | GPS observation identifier |
| `candidate_arc_id` | int | Road segment ID |
| `rf_probability` | float32 | Positive-class emission score [0,1] |
| `selected` | bool | True for the Top-1 predicted arc |

**HMM usage:**
- Use `rf_probability` as the **emission score** for each candidate state
- Use road-network connectivity (`arc_to[a1] == arc_from[a2]`) for **transition scores**
- Do not normalise `rf_probability` values across candidates — they are not a valid categorical distribution

In [76]:
# Cell 95 – Final artifact save
import datetime
all_artifacts = {
    'pipeline':          str(ARTIFACTS_DIR / 'rf_pipeline.joblib'),
    'rf_model':          str(ARTIFACTS_DIR / 'random_forest.joblib'),
    'config_snapshot':   str(ARTIFACTS_DIR / 'rf_config_snapshot.json'),
    'feature_names':     str(ARTIFACTS_DIR / 'feature_names.json'),
}
print('Final artifacts:')
for name, path in all_artifacts.items():
    exists = Path(path).exists()
    print(f'  {name:20s}: {path}  [{"✓" if exists else "✗"}]')

Final artifacts:
  pipeline            : /Users/sandeeph/Desktop/ML_Lab/Random Forest/models/random_forest/artifacts/rf_pipeline.joblib  [✓]
  rf_model            : /Users/sandeeph/Desktop/ML_Lab/Random Forest/models/random_forest/artifacts/random_forest.joblib  [✓]
  config_snapshot     : /Users/sandeeph/Desktop/ML_Lab/Random Forest/models/random_forest/artifacts/rf_config_snapshot.json  [✓]
  feature_names       : /Users/sandeeph/Desktop/ML_Lab/Random Forest/models/random_forest/artifacts/feature_names.json  [✓]


In [77]:
# Cell 96 – Save all results
all_results = {
    'cv_results':          str(RESULTS_DIR / 'cv_results.csv'),
    'oof_predictions':     str(RESULTS_DIR / 'oof_predictions.parquet'),
    'student_test_results':str(RESULTS_DIR / 'student_test_results.csv'),
    'student_test_preds':  str(RESULTS_DIR / 'student_test_predictions.parquet'),
    'feature_importance':  str(RESULTS_DIR / 'feature_importance.csv'),
    'hmm_scores':          str(RESULTS_DIR / 'hmm_candidate_scores.parquet'),
    'error_analysis':      str(RESULTS_DIR / 'error_analysis.csv'),
}

all_metrics_json = {
    'cv_summary': {m: {'mean': float(cv_df[m].mean()), 'std': float(cv_df[m].std())}
                   for m in ['macro_f1','accuracy','precision','recall']
                   if m in cv_df.columns},
    'student_test': {k: v for k,v in all_test_metrics.items() if k != 'confusion_matrix'},
    'final_train_time_s': round(final_train_time, 2),
}
with open(RESULTS_DIR / 'metrics.json', 'w') as f:
    json.dump(all_metrics_json, f, indent=2, default=str)

print('Results:')
for name, path in all_results.items():
    exists = Path(path).exists()
    print(f'  {name:28s}: [{"✓" if exists else "✗"}]')

Results:
  cv_results                  : [✓]
  oof_predictions             : [✓]
  student_test_results        : [✓]
  student_test_preds          : [✓]
  feature_importance          : [✓]
  hmm_scores                  : [✓]
  error_analysis              : [✓]


In [78]:
# Cell 97 – Experiment manifest
manifest = {
    'experiment':        'RF_01_initial_config',
    'timestamp':         datetime.datetime.now().isoformat(),
    'random_state':      RANDOM_STATE,
    'n_folds':           N_FOLDS,
    'threshold':         THRESHOLD,
    'n_training_trips':  train_df['trip_id'].nunique(),
    'n_student_trips':   test_df['trip_id'].nunique(),
    'n_training_cands':  len(train_df),
    'n_student_cands':   len(test_df),
    'rf_params':         {k: str(v) for k,v in RF_INITIAL_PARAMS.items()},
    'feature_cols':      FEATURE_COLS,
    'cv_macro_f1_mean':  round(cv_df['macro_f1'].mean(), 4),
    'student_macro_f1':  test_m['macro_f1'],
    'student_top1_acc':  t1_stats['top1_accuracy'],
    'sklearn_version':   _sklearn.__version__,
    'artifacts':         all_artifacts,
    'results':           all_results,
}
with open(RESULTS_DIR / 'experiment_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)
print(f'Manifest saved → {RESULTS_DIR / "experiment_manifest.json"}')
print('\n=== NOTEBOOK 01 COMPLETE ===')

Manifest saved → /Users/sandeeph/Desktop/ML_Lab/Random Forest/models/random_forest/results/experiment_manifest.json

=== NOTEBOOK 01 COMPLETE ===


## Final Summary

**Random Forest – Notebook 01 complete.**

### Key Results
| Metric | CV Mean ± Std | Student Test |
|---|---|---|
| Macro F1 | (from execution) | (from execution) |
| Top-1 Accuracy | (from execution) | (from execution) |
| Log Loss | (from execution) | (from execution) |
| Route Continuity | (from execution) | (from execution) |

### Next Steps
1. Compare RF results with the other four team models (Decision Tree, SVM, Logistic Regression, etc.)
2. If RF is selected as the final model → run `RF_02_Bayesian_Optimization.py` (Phase 11)
3. Pass `hmm_candidate_scores.parquet` to the HMM/Viterbi stage

### Limitations
- Results depend on the quality of Notebook 00 candidate generation
- Route continuity requires real road-network connectivity arrays (`arcs_from`, `arcs_to`)
- Geographical visualisation requires real GPS coordinates and arc geometries